# Part C – Colab backup run
Same code as the Kaggle notebook `sanya1506/notebook-part-c` (real run, SMOKE = False). Only the first cell (data download) and the last cell (save to Drive) are added.


In [ ]:
# ---- Colab setup (backup for the Kaggle Part C run) ----
# Before running: Runtime -> Change runtime type -> T4 GPU.
# Secrets (key icon in the left bar): add KAGGLE_USERNAME and KAGGLE_KEY (values from your kaggle.json),
# and switch "Notebook access" on for both.
import os, glob, subprocess, zipfile, shutil
from google.colab import userdata, drive
drive.mount("/content/drive")   # asked now, so the final save does not wait for you
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
subprocess.run("pip -q install --upgrade kaggle", shell=True, check=True)
IN = "/kaggle/input"; os.makedirs(IN, exist_ok=True); os.makedirs("/kaggle/working", exist_ok=True)
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
# datasets (downloaded inside Colab only)
if not glob.glob(f"{IN}/ham10000/**/HAM10000_metadata.csv", recursive=True):
    sh(f"kaggle datasets download kmader/skin-cancer-mnist-ham10000 -p {IN}/ham10000 --unzip -q")
if not glob.glob(f"{IN}/dermacon/**/Skin_Metadata.csv", recursive=True):
    sh(f"kaggle datasets download hiro002/dermacon-in-dataset -p {IN}/dermacon --unzip -q")
# outputs of Parts A and B (only their zip files)
for part in ("a", "b"):
    dst = f"{IN}/notebook-part-{part}/skin-cancer-detection"
    if not os.path.isdir(f"{dst}/models"):
        tmp = f"/content/out_{part}"
        sh(f"kaggle kernels output sanya1506/notebook-part-{part} -p {tmp} --file-pattern 'part_{part.upper()}_outputs\.zip$'")
        zipfile.ZipFile(f"{tmp}/part_{part.upper()}_outputs.zip").extractall(dst)
print("checkpoints:", len(glob.glob(f"{IN}/notebook-part-*/skin-cancer-detection/models/*.pth")))


# Part C · Robustness, explanations, DermaCon-IN, gate, sanity checks (report 6.2-6.4, 6.8-6.9)

Needs the outputs of Parts A and B. About 2 h on a T4.

**Kaggle settings (right panel):** Accelerator **GPU T4 x2**, Internet **On**.

**Inputs to add:**
* `skin-cancer-mnist-ham10000` (Kevin Mader) - the HAM10000 images
* your **DermaCon-IN** copy (upload it as a *private* Kaggle dataset; metadata CSV + images)
* the outputs of **Part A** and **Part B**: Add Input → Your Work → Notebooks → pick each notebook

**How to run:** first leave `SMOKE = True` and click **Save Version → Save & Run All** (≈15-20 min, checks every step on tiny data). If it finishes without errors, set `SMOKE = False` and run again. Download `part_C_outputs.zip` from the Output tab.

In [ ]:
SMOKE = False      # <- True for the quick check run, then False for the real run
PART = "C"
import os, sys, glob, json, shutil, subprocess, time, traceback
os.environ["SKIN_SMOKE"] = "1" if SMOKE else "0"
os.environ["SKIN_WORKERS"] = "4"
WORK = "/kaggle/working/skin-cancer-detection"
for d in ("src", "data/splits", "configs", "sample_images"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
os.chdir(WORK)
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - turn on the accelerator")
assert torch.cuda.is_available()

In [ ]:
%%writefile config.py
"""Central configuration: paths, hyperparameters, class names and label mappings.

All paths use pathlib so the project runs unchanged on macOS, Linux and Windows.
"""
from pathlib import Path

# --------------------------------------------------------------------------- paths
ROOT_DIR = Path(__file__).resolve().parent
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = ROOT_DIR / "models"
RESULTS_DIR = ROOT_DIR / "results"
SAMPLE_DIR = ROOT_DIR / "sample_images"
DOCS_DIR = ROOT_DIR / "docs"

METADATA_CSV_NAME = "HAM10000_metadata.csv"
KAGGLE_DATASET = "kmader/skin-cancer-mnist-ham10000"

# --------------------------------------------------------------------------- classes
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_FULL_NAMES = {
    "akiec": "Actinic keratoses / intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}
MALIGNANT_CLASSES = ["mel", "bcc", "akiec"]
BENIGN_CLASSES = ["nv", "bkl", "df", "vasc"]
BINARY_MAP = {c: ("Malignant" if c in MALIGNANT_CLASSES else "Benign") for c in CLASS_NAMES}
NUM_CLASSES = len(CLASS_NAMES)

# --------------------------------------------------------------------------- preprocessing
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
HAIR_KERNEL_SIZE = 17          # blackhat structuring element size
HAIR_THRESHOLD = 12            # threshold on blackhat response to build the hair mask
HAIR_MIN_LENGTH = 40           # keep only long, thin mask components (hairs), not skin texture
INPAINT_RADIUS = 3
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID = (8, 8)

# --------------------------------------------------------------------------- splits
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
SEED = 42

# --------------------------------------------------------------------------- training
AVAILABLE_MODELS = ["efficientnet_b0", "efficientnet_b0_long", "resnet50", "densenet121",
                    "convnext_tiny", "swin_t"]
DEFAULT_MODEL = "efficientnet_b0"
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 4
MIN_EPOCHS = 10                # early stopping can only trigger after this many epochs
NUM_WORKERS = 0                # 0 avoids multiprocessing issues on Windows/macOS
QUICK_SAMPLES_PER_CLASS = 40   # --quick mode subset size
QUICK_EPOCHS = 2


def get_device():
    """CUDA if available, else Apple MPS, else CPU."""
    import torch

    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def model_path(model_name: str, quick: bool = False) -> Path:
    """Checkpoint location, shared by src/train.py and the training notebooks."""
    return MODELS_DIR / f"{model_name}{'_quick' if quick else ''}.pth"


# --------------------------------------------------------------------------- full study (report)
import os as _os

HAM_MODELS = AVAILABLE_MODELS                      # the six HAM10000 models of Table 6.1
ARCH_OF = {"efficientnet_b0_long": "efficientnet_b0"}   # run name -> torchvision architecture
CNN_SETTINGS = dict(lr=3e-4, weight_decay=1e-4, schedule="plateau", epochs=15, patience=4,
                    min_epochs=10, batch_size=32, label_smoothing=0.05, dropout=0.3)
TRANSFORMER_SETTINGS = dict(lr=1e-4, weight_decay=0.05, schedule="cosine", epochs=25, patience=6,
                            min_epochs=0, batch_size=32, label_smoothing=0.05, dropout=0.3)
TRAIN_SETTINGS = {
    "efficientnet_b0": CNN_SETTINGS,
    "efficientnet_b0_long": {**CNN_SETTINGS, "epochs": 30, "patience": 6},
    "resnet50": CNN_SETTINGS,
    "densenet121": CNN_SETTINGS,
    "convnext_tiny": TRANSFORMER_SETTINGS,
    "swin_t": TRANSFORMER_SETTINGS,
}
SEEDS = [0, 1, 2]                                  # every study is repeated with these seeds
STUDY_ARCH = "efficientnet_b0"                     # supporting studies use EfficientNet-B0
IMBALANCE_STRATEGIES = ["none", "sqrt", "inverse", "focal", "sampler"]
ENSEMBLE_RANDOM_VECTORS = 4000
N_BOOTSTRAP = 1000
CROP_MARGIN = 0.15
CROP_MIN_AREA = 0.01
SEG_SIZE = 256
EXPLAIN_IMAGES = 200
EXPLAIN_STEPS = 20
GATE_NEOPLASM_RECALL = 0.90
CORRUPTIONS = {
    "brightness": [0.85, 0.70, 0.55, 0.40, 0.30],
    "gaussian_blur": [0.5, 1.0, 1.5, 2.0, 3.0],
    "jpeg_quality": [80, 60, 40, 20, 10],
}

CONFIGS_DIR = ROOT_DIR / "configs"
DERMACON_DIR = DATA_DIR / "dermacon"               # put the DermaCon-IN download here (any layout)
DDI_DIR = DATA_DIR / "ddi"                         # images + ddi_metadata.csv (optional)
MASK_DIR = DATA_DIR / "ham_masks"                  # HAM10000 lesion masks ISIC_xxxxxxx_segmentation.png
PROCESSED_CROP_DIR = DATA_DIR / "processed_crop"
PROCESSED_DERMACON_DIR = DATA_DIR / "processed_dermacon"
PROCESSED_DERMACON_CROP_DIR = DATA_DIR / "processed_dermacon_crop"
STUDY_DIR = RESULTS_DIR / "study"                  # every number of the report is written below here
PREDS_DIR = RESULTS_DIR / "preds"

# SKIN_SMOKE=1 -> tiny subsets, 1 epoch, few bootstrap samples: checks every code path in minutes
SMOKE = _os.environ.get("SKIN_SMOKE", "0") == "1"
if SMOKE:
    N_BOOTSTRAP = 50
    ENSEMBLE_RANDOM_VECTORS = 50
    EXPLAIN_IMAGES = 8

for _d in (CONFIGS_DIR, DERMACON_DIR, DDI_DIR, MASK_DIR, PROCESSED_CROP_DIR, PROCESSED_DERMACON_DIR,
           PROCESSED_DERMACON_CROP_DIR, STUDY_DIR, PREDS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

for _d in (RAW_DIR, PROCESSED_DIR, SPLITS_DIR, MODELS_DIR, RESULTS_DIR, SAMPLE_DIR, DOCS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile src/preprocess.py
"""Dermoscopic image preprocessing.

Pipeline (identical for training and inference):
    1. Hair removal   - morphological black-hat on the grey image -> hair mask -> Telea inpainting
    2. Enhancement    - CLAHE on the L channel of the LAB colour space
    3. Resize         - 224 x 224 (INTER_AREA)
    4. Normalisation  - scale to [0, 1] then ImageNet mean / std (done in the torch transforms)

Run as a script to cache the preprocessed dataset to data/processed/ (speeds up training):
    python src/preprocess.py
"""
import sys
from pathlib import Path

import cv2
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402


def load_image_rgb(path) -> np.ndarray:
    """Read an image from disk as RGB uint8 (works with non-ASCII Windows paths)."""
    data = np.fromfile(str(path), dtype=np.uint8)
    bgr = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def hair_mask(img_rgb: np.ndarray,
              kernel_size: int = config.HAIR_KERNEL_SIZE,
              threshold: int = config.HAIR_THRESHOLD,
              min_length: int = config.HAIR_MIN_LENGTH):
    """Black-hat response and binary hair mask.

    The black-hat transform responds to thin dark structures. Thresholding alone also catches skin
    texture and the lesion's pigment network, so only connected components that are long
    (bounding box >= min_length px) and sparse (fill ratio < 0.4), i.e. hair-shaped, are kept.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    w, h, area = stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT], stats[:, cv2.CC_STAT_AREA]
    keep = (np.maximum(w, h) >= min_length) & (area / np.maximum(w * h, 1) < 0.4)
    keep[0] = False  # background
    mask = np.where(keep[labels], 255, 0).astype(np.uint8)
    return blackhat, cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)


def remove_hair(img_rgb: np.ndarray, radius: int = config.INPAINT_RADIUS) -> np.ndarray:
    """Remove dark hair strands: black-hat hair mask followed by Telea inpainting."""
    _, mask = hair_mask(img_rgb)
    return cv2.inpaint(img_rgb, mask, radius, cv2.INPAINT_TELEA)


def apply_clahe(img_rgb: np.ndarray,
                clip_limit: float = config.CLAHE_CLIP_LIMIT,
                tile_grid=config.CLAHE_TILE_GRID) -> np.ndarray:
    """Contrast Limited Adaptive Histogram Equalisation on the luminance channel."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l_chan, a_chan, b_chan = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tuple(tile_grid))
    lab = cv2.merge((clahe.apply(l_chan), a_chan, b_chan))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def resize(img_rgb: np.ndarray, size: int = config.IMG_SIZE) -> np.ndarray:
    return cv2.resize(img_rgb, (size, size), interpolation=cv2.INTER_AREA)


def normalize(img_rgb: np.ndarray) -> np.ndarray:
    """uint8 HxWx3 -> float32 HxWx3 normalised with ImageNet statistics."""
    img = img_rgb.astype(np.float32) / 255.0
    return (img - np.array(config.IMAGENET_MEAN, np.float32)) / np.array(config.IMAGENET_STD, np.float32)


def preprocess_image(img_rgb: np.ndarray, size: int = config.IMG_SIZE) -> np.ndarray:
    """Hair removal -> CLAHE -> resize. Returns an RGB uint8 image (size x size).

    Hair removal runs at a fixed working resolution so the morphological kernel
    behaves the same regardless of the input image size.
    """
    h, w = img_rgb.shape[:2]
    scale = 600.0 / max(h, w)
    if abs(scale - 1.0) > 0.05:
        img_rgb = cv2.resize(img_rgb, (int(round(w * scale)), int(round(h * scale))),
                             interpolation=cv2.INTER_AREA)
    img = remove_hair(img_rgb)
    img = apply_clahe(img)
    return resize(img, size)


def preprocess_path(path, size: int = config.IMG_SIZE) -> np.ndarray:
    return preprocess_image(load_image_rgb(path), size)


def save_image_rgb(img_rgb: np.ndarray, path) -> None:
    ok, buf = cv2.imencode(Path(path).suffix or ".jpg", cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR),
                           [cv2.IMWRITE_JPEG_QUALITY, 95])
    if not ok:
        raise ValueError(f"Could not encode image for {path}")
    buf.tofile(str(path))


def cache_dataset(force: bool = False) -> int:
    """Preprocess every HAM10000 image once into data/processed/<image_id>.jpg."""
    from tqdm import tqdm
    from dataset import load_metadata

    df = load_metadata()
    done = 0
    for row in tqdm(df.itertuples(index=False), total=len(df), desc="Preprocessing"):
        out = config.PROCESSED_DIR / f"{row.image_id}.jpg"
        if out.exists() and not force:
            continue
        save_image_rgb(preprocess_path(row.path), out)
        done += 1
    print(f"Preprocessed {done} new images into {config.PROCESSED_DIR}")
    return done


if __name__ == "__main__":
    import argparse

    sys.path.insert(0, str(Path(__file__).resolve().parent))
    parser = argparse.ArgumentParser(description="Cache preprocessed HAM10000 images")
    parser.add_argument("--force", action="store_true", help="re-process existing files")
    cache_dataset(parser.parse_args().force)

In [ ]:
%%writefile src/dataset.py
"""HAM10000 metadata loading, leakage-free splitting and the PyTorch Dataset.

Splits are stratified by diagnosis and grouped by lesion_id: every image of a given
lesion lands in exactly one of train / val / test (70 / 15 / 15).
"""
import sys
from pathlib import Path

import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from preprocess import load_image_rgb, preprocess_path  # noqa: E402

CLASS_TO_IDX = {c: i for i, c in enumerate(config.CLASS_NAMES)}


# --------------------------------------------------------------------------- metadata
def find_metadata_csv() -> Path:
    matches = sorted(config.RAW_DIR.rglob(config.METADATA_CSV_NAME))
    if not matches:
        raise FileNotFoundError(
            f"{config.METADATA_CSV_NAME} not found under {config.RAW_DIR}. "
            "Run: python scripts/download_data.py")
    return matches[0]


def load_metadata() -> pd.DataFrame:
    """Metadata with an absolute `path` column and integer `label`."""
    df = pd.read_csv(find_metadata_csv())
    paths = {}
    for p in config.RAW_DIR.rglob("ISIC_*.jpg"):
        paths.setdefault(p.stem, p)  # the Kaggle archive contains duplicate folders
    df["path"] = df["image_id"].map(paths)
    missing = df["path"].isna().sum()
    if missing:
        print(f"Warning: {missing} images listed in metadata were not found; dropping them.")
        df = df.dropna(subset=["path"])
    df["label"] = df["dx"].map(CLASS_TO_IDX)
    df["binary"] = df["dx"].map(config.BINARY_MAP)
    return df.reset_index(drop=True)


def _resolve_paths(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["path"] = df["path"].map(lambda rel: str(config.RAW_DIR / Path(rel)))
    return df


def make_splits(force: bool = False) -> dict:
    """Create (or load cached) train/val/test splits grouped by lesion_id."""
    files = {s: config.SPLITS_DIR / f"{s}.csv" for s in ("train", "val", "test")}
    if not force and all(f.exists() for f in files.values()):
        return {s: _resolve_paths(pd.read_csv(f)) for s, f in files.items()}

    df = load_metadata()
    lesions = df.groupby("lesion_id")["dx"].first().reset_index()
    train_l, rest_l = train_test_split(
        lesions, test_size=config.VAL_FRAC + config.TEST_FRAC,
        stratify=lesions["dx"], random_state=config.SEED)
    val_l, test_l = train_test_split(
        rest_l, test_size=config.TEST_FRAC / (config.VAL_FRAC + config.TEST_FRAC),
        stratify=rest_l["dx"], random_state=config.SEED)

    splits = {}
    for name, lesion_df in (("train", train_l), ("val", val_l), ("test", test_l)):
        part = df[df["lesion_id"].isin(set(lesion_df["lesion_id"]))].copy()
        # store paths relative to data/raw (POSIX style) so the CSVs are portable across machines/OSes
        part["path"] = part["path"].map(lambda p: Path(p).relative_to(config.RAW_DIR).as_posix())
        part.to_csv(files[name], index=False)
        splits[name] = _resolve_paths(part)
    assert not (set(splits["train"].lesion_id) & set(splits["test"].lesion_id)), "lesion leakage"
    assert not (set(splits["train"].lesion_id) & set(splits["val"].lesion_id)), "lesion leakage"
    for name, part in splits.items():
        print(f"{name:5s}: {len(part):5d} images, {part.lesion_id.nunique():5d} lesions")
    return splits


def quick_subset(df: pd.DataFrame, per_class: int) -> pd.DataFrame:
    shuffled = df.sample(frac=1.0, random_state=config.SEED)
    return shuffled.groupby("dx").head(per_class).reset_index(drop=True)


# --------------------------------------------------------------------------- transforms
def get_transforms(train: bool):
    norm = [transforms.ToTensor(),
            transforms.Normalize(config.IMAGENET_MEAN, config.IMAGENET_STD)]
    if not train:
        return transforms.Compose([transforms.ToPILImage()] + norm)
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),  # zoom
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03),
    ] + norm)


# --------------------------------------------------------------------------- dataset
class SkinLesionDataset(Dataset):
    """Returns (normalised tensor, label). Uses cached preprocessed images when present."""

    def __init__(self, df: pd.DataFrame, train: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = get_transforms(train)

    def __len__(self):
        return len(self.df)

    def load_preprocessed(self, idx: int):
        row = self.df.iloc[idx]
        cached = config.PROCESSED_DIR / f"{row.image_id}.jpg"
        if cached.exists():
            return load_image_rgb(cached)
        return preprocess_path(Path(row.path))

    def __getitem__(self, idx):
        img = self.load_preprocessed(idx)
        return self.transform(img), int(self.df.iloc[idx]["label"])


def get_dataloaders(batch_size: int = config.BATCH_SIZE, quick: bool = False):
    splits = make_splits()
    train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]
    if quick:
        train_df = quick_subset(train_df, config.QUICK_SAMPLES_PER_CLASS)
        val_df = quick_subset(val_df, config.QUICK_SAMPLES_PER_CLASS // 2)
        test_df = quick_subset(test_df, config.QUICK_SAMPLES_PER_CLASS // 2)
    pin = torch.cuda.is_available()
    loaders = {
        "train": DataLoader(SkinLesionDataset(train_df, train=True), batch_size=batch_size,
                            shuffle=True, num_workers=config.NUM_WORKERS, pin_memory=pin),
        "val": DataLoader(SkinLesionDataset(val_df), batch_size=batch_size,
                          shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=pin),
        "test": DataLoader(SkinLesionDataset(test_df), batch_size=batch_size,
                           shuffle=False, num_workers=config.NUM_WORKERS, pin_memory=pin),
    }
    return loaders, {"train": train_df, "val": val_df, "test": test_df}


if __name__ == "__main__":
    s = make_splits(force=True)
    for k, v in s.items():
        print(k, v["dx"].value_counts().to_dict())

In [ ]:
%%writefile src/models.py
"""ImageNet-pretrained backbones (torchvision) with a new head: dropout -> linear.

Six HAM10000 models (report Table 6.1): EfficientNet-B0, EfficientNet-B0 (long run), ResNet50,
DenseNet121, ConvNeXt-Tiny and Swin-T. The same builder makes the 8-class Stage 1 gate.
"""
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402

DISPLAY_NAMES = {
    "efficientnet_b0": "EfficientNet-B0",
    "efficientnet_b0_long": "EfficientNet-B0 (long)",
    "resnet50": "ResNet50",
    "densenet121": "DenseNet121",
    "convnext_tiny": "ConvNeXt-Tiny",
    "swin_t": "Swin-T",
}


def arch_of(name: str) -> str:
    return config.ARCH_OF.get(name, name)


def build_model(name: str, num_classes: int = config.NUM_CLASSES, pretrained: bool = True,
                dropout: float = 0.3) -> nn.Module:
    arch = arch_of(name)
    head = lambda f: nn.Sequential(nn.Dropout(dropout), nn.Linear(f, num_classes))  # noqa: E731
    if arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = head(m.classifier[1].in_features)
    elif arch == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = head(m.fc.in_features)
    elif arch == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = head(m.classifier.in_features)
    elif arch == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[2] = head(m.classifier[2].in_features)
    elif arch == "swin_t":
        m = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1 if pretrained else None)
        m.head = head(m.head.in_features)
    else:
        raise ValueError(f"Unknown model '{name}'. Choose from {config.AVAILABLE_MODELS}")
    return m


def get_target_layer(model: nn.Module, name: str) -> nn.Module:
    """Grad-CAM target: last convolutional stage, output must be (N, C, H, W)."""
    arch = arch_of(name)
    if arch == "efficientnet_b0":
        return model.features[-1]
    if arch == "resnet50":
        return model.layer4[-1]
    if arch == "densenet121":
        # denseblock4 output (norm5 is followed by an in-place ReLU, which breaks tensor hooks)
        return model.features.denseblock4
    if arch == "convnext_tiny":
        return model.features[-1]
    if arch == "swin_t":
        # Swin features are (N, H, W, C); model.permute turns them into (N, C, H, W)
        return model.permute
    raise ValueError(name)


def load_checkpoint(path, device=None):
    """Load a checkpoint saved by the training code. Returns (model, checkpoint_dict)."""
    device = device or config.get_device()
    ckpt = torch.load(str(path), map_location="cpu", weights_only=False)
    n = ckpt.get("num_classes", len(ckpt["class_names"]))
    model = build_model(ckpt.get("arch", ckpt["model_name"]), n, pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    return model, ckpt


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [ ]:
%%writefile src/gradcam.py
"""Grad-CAM (Selvaraju et al., 2017) implemented from scratch with PyTorch hooks.

    alpha_k  = mean_{i,j} dy_c / dA^k_{ij}            (channel importance weights)
    L_c      = ReLU( sum_k alpha_k * A^k )             (class activation map)

Usage:
    python src/gradcam.py --image sample_images/mel_ISIC_0000000.jpg
    python src/gradcam.py --all-samples              # every image in sample_images/
"""
import argparse
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402


class GradCAM:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        self._handle = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach()
        # tensor hook: called with dL/d(output) during backward
        # (skipped when the model is run without gradients, e.g. the deletion/insertion curves under no_grad)
        if output.requires_grad:
            output.register_hook(self._save_grad)

    def _save_grad(self, grad):
        self.gradients = grad.detach()

    def __call__(self, x: torch.Tensor, class_idx: int = None):
        """x: (1,3,H,W). Returns (cam HxW in [0,1], class_idx, probs)."""
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = self.model(x)
            probs = torch.softmax(logits, 1)[0].detach().cpu().numpy()
            if class_idx is None:
                class_idx = int(logits.argmax(1).item())
            logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)            # (1,K,1,1)
        cam = F.relu((weights * self.activations).sum(1, keepdim=True))    # (1,1,h,w)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, probs

    def remove(self):
        self._handle.remove()


def overlay_heatmap(img_rgb: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Blend a JET-coloured CAM over an RGB uint8 image."""
    cam = cv2.resize(cam, (img_rgb.shape[1], img_rgb.shape[0]))
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return np.uint8(np.clip((1 - alpha) * img_rgb + alpha * heat, 0, 255))


def save_figure(original, preprocessed, overlay, title, out_path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.3))
    for ax, im, t in zip(axes, [original, preprocessed, overlay],
                         ["Original", "Preprocessed (hair removal + CLAHE)", "Grad-CAM overlay"]):
        ax.imshow(im)
        ax.set_title(t, fontsize=10)
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)


if __name__ == "__main__":
    from predict import Predictor

    p = argparse.ArgumentParser(description="Generate Grad-CAM visualisations")
    p.add_argument("--image", type=Path, help="path to a dermoscopic image")
    p.add_argument("--all-samples", action="store_true", help="run on every file in sample_images/")
    p.add_argument("--model", default=config.DEFAULT_MODEL, choices=config.AVAILABLE_MODELS)
    p.add_argument("--out-dir", type=Path, default=config.RESULTS_DIR / "gradcam")
    a = p.parse_args()

    images = sorted(config.SAMPLE_DIR.glob("*.jpg")) if a.all_samples else [a.image]
    if not images or images[0] is None:
        p.error("give --image PATH or --all-samples")
    a.out_dir.mkdir(parents=True, exist_ok=True)
    predictor = Predictor(model_name=a.model)
    for img_path in images:
        r = predictor.predict(img_path)
        title = (f"{img_path.name}: predicted {r['pred_class']} ({r['confidence']:.1%}) - "
                 f"{r['binary_label']}")
        out = a.out_dir / f"gradcam_{img_path.stem}.png"
        save_figure(r["original"], r["preprocessed"], r["overlay"], title, out)
        print(f"{title}  ->  {out}")

In [ ]:
%%writefile src/engine.py
"""Generic training / prediction engine used by every experiment in the report.

One function, `train_run`, trains any of the models (HAM10000 models, study models, Stage 1 gate):
  * architecture + settings of report Table 4.2 (AdamW, plateau or warm-up+cosine, early stopping)
  * class-imbalance strategy: none | sqrt | inverse | focal | sampler          (Table 4.3)
  * optional training-only skin-tone augmentation                              (Section 4.7.2)
  * any image set (full HAM images, U-Net crops, DermaCon-IN photos) via a `cache` column
  * best checkpoint by validation macro-F1; finished runs are skipped when re-run (resume)

`predict_probs` gives softmax outputs, optionally averaged over 3 views (original, h-flip, v-flip).
"""
from __future__ import annotations

import json
import math
import os
import random
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
from models import build_model  # noqa: E402
from preprocess import load_image_rgb, preprocess_path, save_image_rgb  # noqa: E402

WORKERS = int(os.environ.get("SKIN_WORKERS", config.NUM_WORKERS))
RUNS_DIR = config.RESULTS_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------- utilities
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def device():
    return config.get_device()


def ensure_cached(df: pd.DataFrame, src_col: str = "path", cache_col: str = "cache",
                  preprocess=preprocess_path, workers: int | None = None) -> None:
    """Preprocess (hair removal, CLAHE, resize) every missing cache file, in parallel."""
    todo = [(s, c) for s, c in zip(df[src_col], df[cache_col]) if not Path(c).exists()]
    if not todo:
        return
    from concurrent.futures import ProcessPoolExecutor
    from tqdm import tqdm
    n = workers or max(1, (os.cpu_count() or 2))
    print(f"preprocessing {len(todo)} images with {n} processes ...")
    with ProcessPoolExecutor(n) as ex:
        list(tqdm(ex.map(_cache_one, todo, chunksize=16), total=len(todo)))


def _cache_one(item):
    src, dst = item
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    save_image_rgb(preprocess_path(src), dst)


def smoke_subset(df: pd.DataFrame, label_col: str, per_class: int) -> pd.DataFrame:
    return df.sample(frac=1.0, random_state=0).groupby(label_col).head(per_class).reset_index(drop=True)


# --------------------------------------------------------------------------- skin-tone augmentation
def shades_of_grey(img: np.ndarray, power: int = 6) -> np.ndarray:
    x = img.astype(np.float32)
    illum = np.power(np.mean(np.power(x, power), axis=(0, 1)), 1.0 / power)
    illum = illum / (np.sqrt((illum ** 2).sum()) + 1e-6)
    return np.clip(x / (illum * np.sqrt(3.0) + 1e-6), 0, 255).astype(np.uint8)


def darker_skin_lab(img: np.ndarray, rng=random) -> np.ndarray:
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
    lab[..., 0] *= rng.uniform(0.55, 0.95)        # lightness x 0.55-0.95
    lab[..., 2] += rng.uniform(0.0, 12.0)         # yellow-blue channel + 0..12
    return cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)


class SkinToneAugment:
    """Training-only: each of the three transforms is applied with probability 0.5 (report Fig. 4.3)."""

    def __init__(self):
        self.jitter = transforms.ColorJitter(brightness=0.25, contrast=0.25, hue=0.05)

    def __call__(self, img: np.ndarray) -> np.ndarray:
        if random.random() < 0.5:
            img = shades_of_grey(img)
        if random.random() < 0.5:
            img = darker_skin_lab(img)
        if random.random() < 0.5:
            from PIL import Image
            img = np.asarray(self.jitter(Image.fromarray(img)))
        return img


# --------------------------------------------------------------------------- data
NORM = [transforms.ToTensor(), transforms.Normalize(config.IMAGENET_MEAN, config.IMAGENET_STD)]


def train_transform():
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03),
    ] + NORM)


def eval_transform():
    return transforms.Compose([transforms.ToPILImage()] + NORM)


class CachedImageDataset(Dataset):
    """Reads preprocessed 224x224 images from df[cache_col]; returns (tensor, label)."""

    def __init__(self, df, label_col="label", cache_col="cache", train=False, skin_aug=False):
        self.paths = df[cache_col].astype(str).tolist()
        self.labels = df[label_col].astype(int).tolist() if label_col in df else [0] * len(df)
        self.tf = train_transform() if train else eval_transform()
        self.skin = SkinToneAugment() if (train and skin_aug) else None

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = load_image_rgb(self.paths[i])
        if img.shape[:2] != (config.IMG_SIZE, config.IMG_SIZE):
            img = cv2.resize(img, (config.IMG_SIZE, config.IMG_SIZE), interpolation=cv2.INTER_AREA)
        if self.skin is not None:
            img = self.skin(img)
        return self.tf(img), self.labels[i]


# --------------------------------------------------------------------------- losses
def balanced_weights(labels: np.ndarray, n_classes: int) -> np.ndarray:
    present = np.unique(labels)
    w = np.ones(n_classes, dtype=np.float32)
    w[present] = compute_class_weight("balanced", classes=present, y=labels)
    return w


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, y):
        logp = F.log_softmax(logits.float(), 1)
        logp_t = logp.gather(1, y[:, None])[:, 0]
        return (-(1 - logp_t.exp()) ** self.gamma * logp_t).mean()


def make_criterion(strategy: str, labels: np.ndarray, n_classes: int, smoothing: float, dev):
    if strategy == "focal":
        return FocalLoss(2.0)
    if strategy in ("none", "sampler"):
        return nn.CrossEntropyLoss(label_smoothing=smoothing)
    w = balanced_weights(labels, n_classes)
    if strategy == "sqrt":
        w = np.sqrt(w)
    elif strategy != "inverse":
        raise ValueError(strategy)
    w = torch.tensor(w / w.mean(), dtype=torch.float32, device=dev)
    return nn.CrossEntropyLoss(weight=w, label_smoothing=smoothing)


# --------------------------------------------------------------------------- training
def run_done(run_name: str) -> bool:
    return (RUNS_DIR / run_name / "summary.json").exists() and (config.MODELS_DIR / f"{run_name}.pth").exists()


def train_run(run_name: str, arch_name: str, train_df: pd.DataFrame, val_df: pd.DataFrame, *,
              n_classes: int = config.NUM_CLASSES, class_names=None, label_col: str = "label",
              cache_col: str = "cache", seed: int = 42, imbalance: str = "sqrt", skin_aug: bool = False,
              settings: dict | None = None, force: bool = False) -> Path:
    """Train one model; returns the checkpoint path. Skips finished runs unless force=True."""
    ckpt_path = config.MODELS_DIR / f"{run_name}.pth"
    out_dir = RUNS_DIR / run_name
    if run_done(run_name) and not force:
        print(f"[skip] {run_name} already trained")
        return ckpt_path
    out_dir.mkdir(parents=True, exist_ok=True)
    s = dict(settings or config.TRAIN_SETTINGS.get(arch_name, config.CNN_SETTINGS))
    if config.SMOKE:
        train_df = smoke_subset(train_df, label_col, 8)
        val_df = smoke_subset(val_df, label_col, 4)
        s.update(epochs=1, patience=1, min_epochs=0)
    set_seed(seed)
    dev = device()
    use_amp = dev.type == "cuda"
    labels = train_df[label_col].to_numpy().astype(int)

    sampler = None
    if imbalance == "sampler":
        counts = np.bincount(labels, minlength=n_classes).astype(float)
        sw = 1.0 / counts[labels]
        sampler = WeightedRandomSampler(torch.tensor(sw, dtype=torch.double), num_samples=len(labels),
                                        replacement=True, generator=torch.Generator().manual_seed(seed))
    g = torch.Generator().manual_seed(seed)
    dl_tr = DataLoader(CachedImageDataset(train_df, label_col, cache_col, train=True, skin_aug=skin_aug),
                       batch_size=s["batch_size"], shuffle=sampler is None, sampler=sampler, generator=g,
                       num_workers=WORKERS, pin_memory=use_amp, drop_last=len(train_df) > s["batch_size"],
                       persistent_workers=WORKERS > 0)
    dl_va = DataLoader(CachedImageDataset(val_df, label_col, cache_col), batch_size=64,
                       num_workers=WORKERS, pin_memory=use_amp)

    model = build_model(arch_name, n_classes, pretrained=True, dropout=s["dropout"]).to(dev)
    crit = make_criterion(imbalance, labels, n_classes, s["label_smoothing"], dev)
    opt = torch.optim.AdamW(model.parameters(), lr=s["lr"], weight_decay=s["weight_decay"])
    steps = s["epochs"] * len(dl_tr)
    if s["schedule"] == "cosine":
        warm = len(dl_tr)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda k: (k + 1) / warm if k < warm else 0.5 * (1 + math.cos(math.pi * (k - warm) / max(1, steps - warm))))
    else:
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.3, patience=2)
    scaler = torch.amp.GradScaler(enabled=use_amp)

    best, bad, hist, t0 = -1.0, 0, [], time.time()
    y_val = val_df[label_col].to_numpy().astype(int)
    for ep in range(1, s["epochs"] + 1):
        model.train()
        tl, n = 0.0, 0
        for x, y in dl_tr:
            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
            with torch.autocast(dev.type, enabled=use_amp):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if s["schedule"] == "cosine":
                sched.step()
            tl += loss.item() * len(y)
            n += len(y)
        pv = predict_loader(model, dl_va, tta=False)
        f1 = f1_score(y_val, pv.argmax(1), average="macro", labels=list(range(n_classes)), zero_division=0)
        acc = float((pv.argmax(1) == y_val).mean())
        if s["schedule"] != "cosine":
            sched.step(f1)
        hist.append({"epoch": ep, "train_loss": tl / max(n, 1), "val_acc": acc, "val_f1": f1,
                     "lr": opt.param_groups[0]["lr"], "minutes": (time.time() - t0) / 60})
        print(f"{run_name} ep {ep}: loss {tl / max(n, 1):.4f} val acc {acc:.4f} macro-F1 {f1:.4f}")
        if f1 > best:
            best, bad = f1, 0
            torch.save({"model_name": run_name, "arch": arch_name, "state_dict": model.state_dict(),
                        "class_names": class_names or config.CLASS_NAMES, "num_classes": n_classes,
                        "img_size": config.IMG_SIZE, "epoch": ep, "val_f1": f1, "val_acc": acc,
                        "seed": seed, "imbalance": imbalance, "skin_aug": skin_aug,
                        "cache_col": cache_col}, ckpt_path)
        else:
            bad += 1
            if bad >= s["patience"] and ep >= s["min_epochs"]:
                print(f"early stop at epoch {ep}")
                break
    pd.DataFrame(hist).to_csv(out_dir / "history.csv", index=False)
    json.dump({"run": run_name, "arch": arch_name, "seed": seed, "imbalance": imbalance,
               "skin_aug": skin_aug, "epochs_run": len(hist), "best_val_f1": best,
               "train_minutes": round((time.time() - t0) / 60, 1), "device": str(dev),
               "n_train": len(train_df), "smoke": config.SMOKE},
              open(out_dir / "summary.json", "w"), indent=2)
    return ckpt_path


# --------------------------------------------------------------------------- prediction
@torch.no_grad()
def predict_loader(model, loader, tta: bool = True) -> np.ndarray:
    """Softmax probabilities; with tta the mean over original, horizontal and vertical flip."""
    model.eval()
    dev = next(model.parameters()).device
    out = []
    for x, _ in loader:
        x = x.to(dev, non_blocking=True)
        views = [x, x.flip(3), x.flip(2)] if tta else [x]
        with torch.autocast(dev.type, enabled=dev.type == "cuda"):
            p = sum(F.softmax(model(v).float(), 1) for v in views) / len(views)
        out.append(p.cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 1))


def predict_df(model, df: pd.DataFrame, cache_col: str = "cache", tta: bool = True,
               batch_size: int = 64) -> np.ndarray:
    dl = DataLoader(CachedImageDataset(df, label_col="__none__", cache_col=cache_col), batch_size=batch_size,
                    num_workers=WORKERS, pin_memory=torch.cuda.is_available())
    return predict_loader(model, dl, tta=tta)


def cached_predictions(key: str, model_or_ckpt, df: pd.DataFrame, cache_col="cache", tta=True,
                       force=False) -> np.ndarray:
    """Predictions saved in results/preds/<key>.npz (with image ids) and reused on later runs."""
    f = config.PREDS_DIR / f"{key}.npz"
    ids = df["image_id"].astype(str).to_numpy()
    if f.exists() and not force:
        z = np.load(f, allow_pickle=True)
        if len(z["ids"]) == len(ids) and (z["ids"] == ids).all():
            return z["probs"]
    model = model_or_ckpt
    if isinstance(model_or_ckpt, (str, Path)):
        from models import load_checkpoint
        model, _ = load_checkpoint(model_or_ckpt)
    p = predict_df(model, df, cache_col, tta)
    np.savez_compressed(f, probs=p, ids=ids)
    return p

In [ ]:
%%writefile src/stats.py
"""Metrics and statistics used throughout the report (Section 4.8).

* `metrics`          - accuracy, balanced accuracy, macro-F1, melanoma recall, malignant sensitivity /
                       specificity / AUC, macro AUC
* `bootstrap_ci`     - 1,000 resamples of GROUPS (lesions for HAM10000, patients for DermaCon-IN)
* `seed_summary`     - mean +/- std over seeds and a bootstrap CI of the seed-mean
* `paired_diff`      - difference B - A with the SAME bootstrap resample for both (paired)
* `wilson`           - Wilson score interval for a proportion
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402

MAL = [config.CLASS_NAMES.index(c) for c in config.MALIGNANT_CLASSES]
MEL = config.CLASS_NAMES.index("mel")
K = config.NUM_CLASSES


def _safe_auc(y, s):
    return float(roc_auc_score(y, s)) if 0 < np.sum(y) < len(y) else float("nan")


def metrics(y: np.ndarray, p: np.ndarray) -> dict:
    y = np.asarray(y).astype(int)
    pred = p.argmax(1)
    rec = [np.mean(pred[y == c] == c) if np.any(y == c) else np.nan for c in range(K)]
    ym, pm = np.isin(y, MAL), np.isin(pred, MAL)
    aucs = [_safe_auc(y == c, p[:, c]) for c in range(K)]
    return {
        "accuracy": float(np.mean(pred == y)),
        "balanced_accuracy": float(np.nanmean(rec)),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=list(range(K)), zero_division=0)),
        "melanoma_recall": float(rec[MEL]) if not np.isnan(rec[MEL]) else float("nan"),
        "malignant_sensitivity": float(pm[ym].mean()) if ym.any() else float("nan"),
        "malignant_specificity": float((~pm[~ym]).mean()) if (~ym).any() else float("nan"),
        "malignant_auc": _safe_auc(ym, p[:, MAL].sum(1)),
        "macro_auc": float(np.nanmean(aucs)),
    }


REPORT_METRICS = ["accuracy", "balanced_accuracy", "macro_f1", "melanoma_recall",
                  "malignant_sensitivity", "malignant_specificity", "malignant_auc"]


def group_index(groups):
    """List of index arrays, one per group (lesion / patient)."""
    groups = np.asarray(groups)
    order = np.argsort(groups, kind="stable")
    g_sorted = groups[order]
    cuts = np.flatnonzero(g_sorted[1:] != g_sorted[:-1]) + 1
    return np.split(order, cuts)


def resamples(groups, n=None, seed=0):
    """Yield index arrays of group-level bootstrap resamples."""
    n = n or config.N_BOOTSTRAP
    idx = group_index(groups)
    rng = np.random.default_rng(seed)
    for _ in range(n):
        pick = rng.integers(0, len(idx), len(idx))
        yield np.concatenate([idx[i] for i in pick])


def bootstrap_ci(fn, groups, n=None, seed=0, alpha=0.05):
    """fn(index_array) -> float. Percentile CI over group resamples."""
    vals = np.array([fn(ix) for ix in resamples(groups, n, seed)], dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 100 * alpha / 2)), float(np.percentile(vals, 100 * (1 - alpha / 2))))


def seed_summary(y, probs_per_seed: list, groups, metric_names=REPORT_METRICS, seed=0) -> dict:
    """{metric: {mean, std, ci: [lo, hi], per_seed: [...]}}; the CI is for the mean over seeds."""
    per = [metrics(y, p) for p in probs_per_seed]
    out = {}
    boot = {m: [] for m in metric_names}
    for ix in resamples(groups, seed=seed):
        ms = [metrics(y[ix], p[ix]) for p in probs_per_seed]
        for m in metric_names:
            boot[m].append(np.nanmean([d[m] for d in ms]))
    for m in metric_names:
        v = np.array([d[m] for d in per], float)
        b = np.array(boot[m], float)
        b = b[~np.isnan(b)]
        out[m] = {"mean": float(np.nanmean(v)), "std": float(np.nanstd(v, ddof=1)) if len(v) > 1 else 0.0,
                  "ci": [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))] if len(b) else [None, None],
                  "per_seed": v.tolist()}
    return out


def paired_diff(y, probs_a: list, probs_b: list, groups, metric_names=REPORT_METRICS, seed=0) -> dict:
    """Mean over seeds of (B - A) with a paired bootstrap CI (same resample for A and B)."""
    point = {}
    for m in metric_names:
        a = np.nanmean([metrics(y, p)[m] for p in probs_a])
        b = np.nanmean([metrics(y, p)[m] for p in probs_b])
        point[m] = b - a
    boot = {m: [] for m in metric_names}
    for ix in resamples(groups, seed=seed):
        ma = [metrics(y[ix], p[ix]) for p in probs_a]
        mb = [metrics(y[ix], p[ix]) for p in probs_b]
        for m in metric_names:
            boot[m].append(np.nanmean([d[m] for d in mb]) - np.nanmean([d[m] for d in ma]))
    return {m: {"diff": float(point[m]),
                "ci": [float(np.nanpercentile(boot[m], 2.5)), float(np.nanpercentile(boot[m], 97.5))]}
            for m in metric_names}


def wilson(k: int, n: int, z: float = 1.959964) -> tuple:
    if n == 0:
        return (float("nan"), float("nan"))
    ph = k / n
    den = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / den
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return (float(max(0.0, c - h)), float(min(1.0, c + h)))


def fmt_ms(d: dict, digits=3) -> str:
    """'0.695 ± 0.023 [0.641, 0.738]' from a seed_summary entry."""
    lo, hi = d["ci"]
    return f"{d['mean']:.{digits}f} ± {d['std']:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"

In [ ]:
%%writefile src/ham.py
"""Part A of the report: six HAM10000 models, 3-view TTA, ensemble selection on validation,
Tables 6.1 / 6.2 / 6.16 and Figures 6.1-6.4 / 6.9.

    from ham import *; prepare(); train_all(); predict_all(); build_ensemble(); figures()
"""
from __future__ import annotations

import itertools
import json
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
from sklearn.metrics import confusion_matrix, f1_score, roc_curve, auc as sk_auc  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
import engine  # noqa: E402
import stats  # noqa: E402
from dataset import make_splits  # noqa: E402
from models import DISPLAY_NAMES, count_parameters, get_target_layer, load_checkpoint  # noqa: E402

OUT = config.STUDY_DIR / "ham"
OUT.mkdir(parents=True, exist_ok=True)
CLASSES = config.CLASS_NAMES


# --------------------------------------------------------------------------- data
def splits(with_cache=True) -> dict:
    s = make_splits()
    for k, df in s.items():
        df = df.copy()
        df["cache"] = df["image_id"].map(lambda i: str(config.PROCESSED_DIR / f"{i}.jpg"))
        df["crop_cache"] = df["image_id"].map(lambda i: str(config.PROCESSED_CROP_DIR / f"{i}.jpg"))
        if config.SMOKE:
            df = engine.smoke_subset(df, "label", 12 if k == "train" else 6)
        s[k] = df.reset_index(drop=True)
    return s


def prepare():
    s = splits()
    for df in s.values():
        engine.ensure_cached(df, "path", "cache")
    t = pd.DataFrame({k: v["dx"].value_counts() for k, v in s.items()}).reindex(CLASSES).fillna(0).astype(int)
    t["total"] = t.sum(1)
    t.loc["images"] = t.sum()
    t.loc["lesions"] = [s[k].lesion_id.nunique() for k in ("train", "val", "test")] + [
        sum(s[k].lesion_id.nunique() for k in ("train", "val", "test"))]
    t.loc["malignant"] = [s[k].dx.isin(config.MALIGNANT_CLASSES).sum() for k in ("train", "val", "test")] + [
        sum(s[k].dx.isin(config.MALIGNANT_CLASSES).sum() for k in ("train", "val", "test"))]
    t.to_csv(OUT / "table_3_1_split.csv")
    print(t)
    return s


# --------------------------------------------------------------------------- training
def train_all(models=None):
    s = splits()
    for m in models or config.HAM_MODELS:
        engine.train_run(m, m, s["train"], s["val"], seed=config.SEED, imbalance="sqrt",
                         settings=config.TRAIN_SETTINGS[m])


def ckpt(m):
    return config.MODELS_DIR / f"{m}.pth"


def predict_all(models=None):
    s = splits()
    out = {}
    for m in models or config.HAM_MODELS:
        out[m] = {k: engine.cached_predictions(f"ham_{m}_{k}", ckpt(m), s[k]) for k in ("val", "test")}
    return out


def load_preds(models=None):
    s = splits()
    return {m: {k: engine.cached_predictions(f"ham_{m}_{k}", ckpt(m), s[k]) for k in ("val", "test")}
            for m in (models or config.HAM_MODELS)}


# --------------------------------------------------------------------------- ensemble
def _f1(y, p):
    return f1_score(y, p.argmax(1), average="macro", labels=list(range(len(CLASSES))), zero_division=0)


def build_ensemble():
    s = splits()
    yv, yt = s["val"].label.to_numpy(), s["test"].label.to_numpy()
    P = load_preds()
    models = list(P)
    V = np.stack([P[m]["val"] for m in models])
    T = np.stack([P[m]["test"] for m in models])

    cands = {"equal": np.ones(len(models)) / len(models)}
    best3, best3_f1 = None, -1
    for combo in itertools.combinations(range(len(models)), 3):
        w = np.zeros(len(models)); w[list(combo)] = 1 / 3
        f = _f1(yv, np.tensordot(w, V, 1))
        if f > best3_f1:
            best3, best3_f1 = w, f
    cands["best3"] = best3
    rng = np.random.default_rng(42)
    bw, bf = None, -1
    for _ in range(config.ENSEMBLE_RANDOM_VECTORS):
        w = rng.random(len(models)); w /= w.sum()
        f = _f1(yv, np.tensordot(w, V, 1))
        if f > bf:
            bw, bf = w, f
    cands["weighted"] = bw
    val_f1 = {k: _f1(yv, np.tensordot(w, V, 1)) for k, w in cands.items()}
    chosen = max(val_f1, key=val_f1.get)              # selection on VALIDATION only
    w = cands[chosen]
    np.savez(OUT / "ensemble_probs.npz", val=np.tensordot(w, V, 1), test=np.tensordot(w, T, 1))
    json.dump({"models": models, "candidates": {k: v.tolist() for k, v in cands.items()},
               "val_macro_f1": val_f1, "chosen": chosen, "weights": w.tolist()},
              open(OUT / "ensemble_choice.json", "w"), indent=2)
    print("validation macro-F1:", val_f1, "-> chosen:", chosen)
    return chosen, w


def ensemble_probs(split="test"):
    return np.load(OUT / "ensemble_probs.npz")[split]


def ensemble_weights():
    c = json.load(open(OUT / "ensemble_choice.json"))
    return dict(zip(c["models"], c["weights"])), c


# --------------------------------------------------------------------------- tables
def time_per_image(m, n=100):
    """Milliseconds per image with 3-view TTA, batch size 1."""
    s = splits()
    model, _ = load_checkpoint(ckpt(m))
    df = s["test"].head(min(n, len(s["test"])))
    dl = torch.utils.data.DataLoader(engine.CachedImageDataset(df), batch_size=1)
    xs = [x for x, _ in dl]
    dev = next(model.parameters()).device
    with torch.no_grad():
        for x in xs[:5]:
            model(x.to(dev))
        if dev.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for x in xs:
            x = x.to(dev)
            for v in (x, x.flip(3), x.flip(2)):
                model(v)
        if dev.type == "cuda":
            torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / len(xs)


def tables():
    s = splits()
    yt, gt = s["test"].label.to_numpy(), s["test"].lesion_id.to_numpy()
    yv = s["val"].label.to_numpy()
    P = load_preds()
    wmap, choice = ensemble_weights()
    rows, params, ms = [], {}, {}
    for m in P:
        model, _ = load_checkpoint(ckpt(m))
        params[m] = count_parameters(model) / 1e6
        ms[m] = time_per_image(m)
        del model
        r = stats.metrics(yt, P[m]["test"])
        rows.append({"model": DISPLAY_NAMES[m], "key": m, "params_m": params[m], "val_f1": _f1(yv, P[m]["val"]),
                     **r, "ms_per_img": ms[m]})
    V = np.stack([P[m]["val"] for m in P]); T = np.stack([P[m]["test"] for m in P])
    for k, label in (("equal", "Ensemble, equal weights"), ("best3", "Ensemble, best 3"),
                     ("weighted", "Ensemble, weighted")):
        w = np.array(choice["candidates"][k])
        used = [m for m, wi in zip(P, w) if wi > 0]
        r = stats.metrics(yt, np.tensordot(w, T, 1))
        rows.append({"model": label + (" (selected)" if k == choice["chosen"] else ""), "key": f"ens_{k}",
                     "params_m": sum(params[m] for m in used), "val_f1": _f1(yv, np.tensordot(w, V, 1)),
                     **r, "ms_per_img": sum(ms[m] for m in used)})
    t61 = pd.DataFrame(rows)
    t61.to_csv(OUT / "table_6_1_models.csv", index=False)

    pe = ensemble_probs("test")
    pred = pe.argmax(1)
    cm = confusion_matrix(yt, pred, labels=range(len(CLASSES)))
    per = []
    for c, name in enumerate(CLASSES):
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c].sum() - tp; tn = cm.sum() - tp - fp - fn
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        per.append({"class": name, "precision": prec, "recall": rec, "specificity": tn / max(tn + fp, 1),
                    "f1": 2 * prec * rec / max(prec + rec, 1e-9), "auc": stats._safe_auc(yt == c, pe[:, c]),
                    "test_images": int((yt == c).sum())})
    pd.DataFrame(per).to_csv(OUT / "table_6_2_per_class.csv", index=False)
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "confusion_matrix.csv")

    miss = []
    for name in config.MALIGNANT_CLASSES:
        c = CLASSES.index(name)
        sel = yt == c
        k = int((~np.isin(pred[sel], stats.MAL)).sum()); n = int(sel.sum())
        lo, hi = stats.wilson(k, n)
        miss.append({"class": name, "test_images": n, "called_benign": k, "miss_rate": k / max(n, 1),
                     "wilson_lo": lo, "wilson_hi": hi})
    pd.DataFrame(miss).to_csv(OUT / "table_6_16_missed_cancers.csv", index=False)

    sel = choice["chosen"]
    summary = {"selected": sel, "test": stats.metrics(yt, pe),
               "test_ci": {m: stats.bootstrap_ci(lambda ix: stats.metrics(yt[ix], pe[ix])[m], gt)
                           for m in ("accuracy", "macro_f1", "melanoma_recall")},
               "val_macro_f1": choice["val_macro_f1"]}
    json.dump(summary, open(OUT / "ensemble_summary.json", "w"), indent=2, default=float)
    print(t61.round(3).to_string(index=False))
    return t61


# --------------------------------------------------------------------------- figures
def ensemble_cam(img_tensor, class_idx=None):
    """Weighted mean of the members' Grad-CAMs for the ensemble's predicted class."""
    from gradcam import GradCAM
    wmap, _ = ensemble_weights()
    cams, probs = [], 0
    for m, w in wmap.items():
        if w <= 0:
            continue
        model, _ = load_checkpoint(ckpt(m))
        x = img_tensor.to(next(model.parameters()).device)
        with torch.no_grad():
            probs = probs + w * torch.softmax(model(x), 1)[0].cpu().numpy()
        cams.append((m, w))
        del model
    cls = int(np.argmax(probs)) if class_idx is None else class_idx
    total = 0
    for m, w in cams:
        model, _ = load_checkpoint(ckpt(m))
        gc = GradCAM(model, get_target_layer(model, m))
        cam, _, _ = gc(img_tensor.to(next(model.parameters()).device), class_idx=cls)
        gc.remove()
        total = total + w * cam
        del model
    total = (total - total.min()) / (total.max() - total.min() + 1e-8)
    return total, cls, probs


def figures():
    from gradcam import overlay_heatmap
    from preprocess import load_image_rgb
    s = splits()
    yt = s["test"].label.to_numpy()
    t61 = pd.read_csv(OUT / "table_6_1_models.csv")

    # Fig 6.1 - metrics of all models and ensembles
    mets = ["accuracy", "balanced_accuracy", "macro_f1", "melanoma_recall", "malignant_sensitivity"]
    fig, ax = plt.subplots(figsize=(13, 4.8))
    x = np.arange(len(mets)); wbar = 0.8 / len(t61)
    for i, r in t61.iterrows():
        ax.bar(x + i * wbar, [r[m] for m in mets], wbar, label=r["model"],
               hatch="//" if str(r["key"]).startswith("ens") else None, edgecolor="white")
    ax.set_xticks(x + 0.4 - wbar / 2); ax.set_xticklabels([m.replace("_", " ") for m in mets])
    ax.set_ylim(0, 1); ax.legend(fontsize=7, ncol=3); ax.grid(axis="y", alpha=.3)
    ax.set_title("HAM10000 test metrics (hatched: ensembles)")
    fig.tight_layout(); fig.savefig(OUT / "fig_6_1_models.png", dpi=150); plt.close(fig)

    # Fig 6.2 - confusion matrix; Fig 6.3 - ROC
    pe = ensemble_probs("test")
    cm = confusion_matrix(yt, pe.argmax(1), labels=range(len(CLASSES)))
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks(range(7)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(7)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.colorbar(im)
    ax.set_title("Selected ensemble - HAM10000 test")
    fig.tight_layout(); fig.savefig(OUT / "fig_6_2_confusion.png", dpi=150); plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for c, name in enumerate(CLASSES):
        if 0 < (yt == c).sum() < len(yt):
            fpr, tpr, _ = roc_curve(yt == c, pe[:, c])
            axes[0].plot(fpr, tpr, label=f"{name} (AUC {sk_auc(fpr, tpr):.3f})")
    ym = np.isin(yt, stats.MAL)
    fpr, tpr, _ = roc_curve(ym, pe[:, stats.MAL].sum(1))
    axes[1].plot(fpr, tpr, color="crimson", label=f"malignant vs benign (AUC {sk_auc(fpr, tpr):.3f})")
    for a in axes:
        a.plot([0, 1], [0, 1], "k--", lw=.8); a.legend(fontsize=8); a.set_xlabel("False positive rate")
        a.set_ylabel("True positive rate"); a.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUT / "fig_6_3_roc.png", dpi=150); plt.close(fig)

    # Fig 6.4 - ensemble Grad-CAM, one test image per class
    tf = engine.eval_transform()
    picks = s["test"].groupby("dx").head(1).sort_values("dx")
    fig, axes = plt.subplots(2, len(picks), figsize=(2.6 * len(picks), 5.6))
    for i, r in enumerate(picks.itertuples()):
        img = load_image_rgb(r.cache)
        cam, cls, probs = ensemble_cam(tf(img)[None])
        axes[0, i].imshow(img); axes[0, i].set_title(f"true {r.dx}", fontsize=9)
        axes[1, i].imshow(overlay_heatmap(img, cam))
        axes[1, i].set_title(f"pred {CLASSES[cls]} {probs[cls]:.2f}", fontsize=9)
        axes[0, i].axis("off"); axes[1, i].axis("off")
    fig.tight_layout(); fig.savefig(OUT / "fig_6_4_gradcam.png", dpi=150); plt.close(fig)

    # Fig 6.9 - most confident missed cancers
    pred = pe.argmax(1)
    missed = np.flatnonzero(np.isin(yt, stats.MAL) & ~np.isin(pred, stats.MAL))
    missed = missed[np.argsort(-pe[missed].max(1))][:6]
    if len(missed):
        fig, axes = plt.subplots(2, len(missed), figsize=(2.6 * len(missed), 5.6), squeeze=False)
        for i, ix in enumerate(missed):
            r = s["test"].iloc[ix]
            img = load_image_rgb(r.cache)
            cam, cls, probs = ensemble_cam(tf(img)[None])
            axes[0, i].imshow(img); axes[0, i].set_title(f"true {r.dx}", fontsize=9)
            axes[1, i].imshow(overlay_heatmap(img, cam))
            axes[1, i].set_title(f"pred {CLASSES[cls]} {probs[cls]:.2f}", fontsize=9)
            axes[0, i].axis("off"); axes[1, i].axis("off")
        fig.tight_layout(); fig.savefig(OUT / "fig_6_9_missed.png", dpi=150); plt.close(fig)
    print("figures ->", OUT)

In [ ]:
%%writefile src/studies.py
"""Supporting studies on HAM10000 (report Sections 4.7 and 6.5-6.9).
All use EfficientNet-B0 + the HAM10000 recipe, seeds 0/1/2, single-view evaluation.

  imbalance_study()      Table 6.10, Fig 6.7   (5 strategies x 3 seeds)
  skin_tone_study()      Table 6.11, Fig 4.3   (selected setup + skin-tone augmentation)
  segmentation_study()   Table 6.12, Fig 4.4   (U-Net, EfficientNet-B0 encoder, BCE+Dice; Otsu baseline)
  crop_study()           Table 6.13            (selected setup trained on predicted-mask crops)
  robustness_study()     Table 6.14, Fig 6.8   (brightness / blur / JPEG, severities 1-5)
  explanation_study()    Table 6.15            (Grad-CAM deletion / insertion, 200 test images)
"""
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import torch  # noqa: E402
import torch.nn as nn  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from torch.utils.data import DataLoader, Dataset  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
import engine  # noqa: E402
import ham  # noqa: E402
import stats  # noqa: E402
from models import get_target_layer, load_checkpoint  # noqa: E402
from preprocess import load_image_rgb, preprocess_image, save_image_rgb  # noqa: E402

OUT = config.STUDY_DIR / "studies"
OUT.mkdir(parents=True, exist_ok=True)
ARCH = config.STUDY_ARCH
NAMES = {"none": "a) None", "sqrt": "b) sqrt balanced weights (default)", "inverse": "c) Inverse frequency",
         "focal": "d) Focal loss", "sampler": "e) Weighted sampler"}


def _run(prefix, seed):
    return f"study_{prefix}_s{seed}"


def _preds(run, df, split, cache_col="cache"):
    return engine.cached_predictions(f"{run}_{split}", config.MODELS_DIR / f"{run}.pth", df, cache_col, tta=False)


# =========================================================================== class imbalance
def imbalance_study():
    s = ham.splits()
    for strat in config.IMBALANCE_STRATEGIES:
        for seed in config.SEEDS:
            engine.train_run(_run(f"imb_{strat}", seed), ARCH, s["train"], s["val"], seed=seed, imbalance=strat)
    yv, yt, gt = s["val"].label.to_numpy(), s["test"].label.to_numpy(), s["test"].lesion_id.to_numpy()
    rows, val = [], {}
    for strat in config.IMBALANCE_STRATEGIES:
        pv = [_preds(_run(f"imb_{strat}", sd), s["val"], "val") for sd in config.SEEDS]
        pt = [_preds(_run(f"imb_{strat}", sd), s["test"], "test") for sd in config.SEEDS]
        mv = [stats.metrics(yv, p) for p in pv]
        val[strat] = (np.mean([m["macro_f1"] for m in mv]), np.mean([m["melanoma_recall"] for m in mv]))
        ts = stats.seed_summary(yt, pt, gt)
        rows.append({"strategy": strat, "label": NAMES[strat], "val_macro_f1": val[strat][0],
                     "val_mel_recall": val[strat][1], **{f"test_{k}": v for k, v in ts.items()}})
    # rule fixed in advance: best mean val macro-F1; within 0.005 -> higher val melanoma recall
    best_f1 = max(v[0] for v in val.values())
    close = [k for k, v in val.items() if best_f1 - v[0] <= 0.005]
    selected = max(close, key=lambda k: (val[k][1], val[k][0]))
    json.dump({"selected": selected, "val": {k: list(v) for k, v in val.items()}},
              open(OUT / "imbalance_selected.json", "w"), indent=2)
    json.dump(rows, open(OUT / "table_6_10_imbalance.json", "w"), indent=2, default=float)
    flat = [{"strategy": r["label"] + (" (selected)" if r["strategy"] == selected else ""),
             "val_macro_f1": r["val_macro_f1"], "val_mel_recall": r["val_mel_recall"],
             "test_macro_f1": stats.fmt_ms(r["test_macro_f1"]), "test_mel_recall": stats.fmt_ms(r["test_melanoma_recall"]),
             "test_mal_sens": stats.fmt_ms(r["test_malignant_sensitivity"]),
             "test_accuracy": f"{r['test_accuracy']['mean']:.3f} ± {r['test_accuracy']['std']:.3f}"} for r in rows]
    pd.DataFrame(flat).to_csv(OUT / "table_6_10_imbalance.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, m in zip(axes, ["macro_f1", "melanoma_recall", "malignant_sensitivity"]):
        for i, r in enumerate(rows):
            d = r[f"test_{m}"]
            ax.errorbar(i, d["mean"], yerr=[[d["mean"] - d["ci"][0]], [d["ci"][1] - d["mean"]]], fmt="o", capsize=4,
                        color="crimson" if r["strategy"] == selected else "steelblue")
        ax.set_xticks(range(len(rows))); ax.set_xticklabels([r["strategy"] for r in rows])
        ax.set_title(m.replace("_", " ")); ax.grid(alpha=.3)
    fig.suptitle("Class-imbalance strategies, HAM10000 test (mean, 95% CI; red = selected on validation)")
    fig.tight_layout(); fig.savefig(OUT / "fig_6_7_imbalance.png", dpi=150); plt.close(fig)
    print(pd.DataFrame(flat).to_string(index=False), "\nselected:", selected)
    return selected


def selected_setup() -> str:
    return json.load(open(OUT / "imbalance_selected.json"))["selected"]


def baseline_runs():
    return [_run(f"imb_{selected_setup()}", sd) for sd in config.SEEDS]


# =========================================================================== skin tone
def skin_tone_study():
    s = ham.splits()
    sel = selected_setup()
    for seed in config.SEEDS:
        engine.train_run(_run("skin", seed), ARCH, s["train"], s["val"], seed=seed, imbalance=sel, skin_aug=True)
    yt, gt = s["test"].label.to_numpy(), s["test"].lesion_id.to_numpy()
    base = [_preds(r, s["test"], "test") for r in baseline_runs()]
    aug = [_preds(_run("skin", sd), s["test"], "test") for sd in config.SEEDS]
    _paired_table(yt, gt, base, aug, "Baseline (selected setup)", "Augmented", OUT / "table_6_11_skin_tone")

    img = load_image_rgb(s["train"].cache.iloc[0])
    random.seed(1)
    views = [("original", img), ("Shades of Grey", engine.shades_of_grey(img)),
             ("darker skin (LAB)", engine.darker_skin_lab(img)),
             ("brightness/contrast/hue", np.asarray(engine.SkinToneAugment().jitter(
                 __import__("PIL.Image", fromlist=["Image"]).fromarray(img))))]
    fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
    for a, (t, v) in zip(axes, views):
        a.imshow(v); a.set_title(t, fontsize=9); a.axis("off")
    fig.tight_layout(); fig.savefig(OUT / "fig_4_3_skin_tone_aug.png", dpi=150); plt.close(fig)


def _paired_table(y, groups, probs_a, probs_b, name_a, name_b, out_stem):
    sa = stats.seed_summary(y, probs_a, groups)
    sb = stats.seed_summary(y, probs_b, groups)
    dd = stats.paired_diff(y, probs_a, probs_b, groups)
    rows = [{"metric": m.replace("_", " "), name_a: stats.fmt_ms(sa[m]), name_b: stats.fmt_ms(sb[m]),
             "difference [95% paired CI]": f"{dd[m]['diff']:+.4f} [{dd[m]['ci'][0]:+.4f}, {dd[m]['ci'][1]:+.4f}]"}
            for m in stats.REPORT_METRICS]
    pd.DataFrame(rows).to_csv(f"{out_stem}.csv", index=False)
    json.dump({"a": sa, "b": sb, "diff": dd}, open(f"{out_stem}.json", "w"), indent=2, default=float)
    print(pd.DataFrame(rows).to_string(index=False))


# =========================================================================== segmentation
class UNetEffB0(nn.Module):
    """U-Net decoder on an ImageNet EfficientNet-B0 encoder (torchvision)."""
    SKIPS = (1, 2, 3, 5)          # feature indices: 16@/2, 24@/4, 40@/8, 112@/16 ; bottleneck 1280@/32

    def __init__(self, pretrained=True):
        super().__init__()
        from torchvision import models
        self.enc = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None).features
        ch = [16, 24, 40, 112]

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
                                 nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
        self.d4 = block(1280 + ch[3], 256)
        self.d3 = block(256 + ch[2], 128)
        self.d2 = block(128 + ch[1], 64)
        self.d1 = block(64 + ch[0], 32)
        self.d0 = block(32, 16)
        self.out = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        feats = {}
        h = x
        for i, layer in enumerate(self.enc):
            h = layer(h)
            if i in self.SKIPS:
                feats[i] = h
        for dec, i in ((self.d4, 5), (self.d3, 3), (self.d2, 2), (self.d1, 1)):
            h = F.interpolate(h, size=feats[i].shape[-2:], mode="bilinear", align_corners=False)
            h = dec(torch.cat([h, feats[i]], 1))
        h = F.interpolate(h, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.out(self.d0(h))


def find_masks() -> dict:
    """image_id -> mask path, searched in data/ham_masks and /kaggle/input."""
    roots = [config.MASK_DIR] + ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    out = {}
    for r in roots:
        for p in r.rglob("ISIC_*_segmentation.png"):
            out.setdefault(p.stem[:12], p)
    return out


def _seg_input(img):
    x = cv2.resize(img, (config.SEG_SIZE, config.SEG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255
    x = (x - np.array(config.IMAGENET_MEAN, np.float32)) / np.array(config.IMAGENET_STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1))


class SegDS(Dataset):
    def __init__(self, df, masks, train):
        self.df, self.masks, self.train = df.reset_index(drop=True), masks, train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = load_image_rgb(r.path)
        m = (cv2.imread(str(self.masks[r.image_id]), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        m = cv2.resize(m, (config.SEG_SIZE, config.SEG_SIZE), interpolation=cv2.INTER_NEAREST)
        x = _seg_input(img)
        y = torch.from_numpy(m[None].astype(np.float32))
        if self.train:
            if random.random() < .5: x, y = x.flip(2), y.flip(2)
            if random.random() < .5: x, y = x.flip(1), y.flip(1)
            k = random.randint(0, 3); x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (1, 2))
        return x, y


def _dice_iou(pred, gt):
    inter = (pred & gt).sum(); s = pred.sum() + gt.sum(); u = (pred | gt).sum()
    return (2 * inter / s if s else 1.0), (inter / u if u else 1.0)


def train_unet(seed, s, masks, epochs=20):
    name = f"unet_s{seed}"
    path = config.MODELS_DIR / f"{name}.pth"
    if path.exists() and (engine.RUNS_DIR / name / "summary.json").exists():
        return path
    engine.set_seed(seed)
    dev = engine.device()
    tr = s["train"][s["train"].image_id.isin(masks)]
    va = s["val"][s["val"].image_id.isin(masks)]
    if config.SMOKE:
        tr, va, epochs = tr.head(16), va.head(8), 1
    dl = DataLoader(SegDS(tr, masks, True), batch_size=16, shuffle=True, num_workers=engine.WORKERS, drop_last=True)
    dv = DataLoader(SegDS(va, masks, False), batch_size=32, num_workers=engine.WORKERS)
    model = UNetEffB0().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = dev.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    best = -1
    for ep in range(epochs):
        model.train()
        for x, y in dl:
            x, y = x.to(dev), y.to(dev)
            with torch.autocast(dev.type, enabled=use_amp):
                o = model(x)
            o = o.float()
            p = torch.sigmoid(o)
            dice = 1 - (2 * (p * y).sum((1, 2, 3)) + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
            loss = F.binary_cross_entropy_with_logits(o, y) + dice.mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        model.eval(); ds = []
        with torch.no_grad():
            for x, y in dv:
                pr = (model(x.to(dev)).float() > 0).cpu().numpy().astype(bool)
                for a, b in zip(pr, y.numpy().astype(bool)):
                    ds.append(_dice_iou(a, b)[0])
        d = float(np.mean(ds))
        print(f"{name} ep {ep + 1}: val Dice {d:.4f}")
        if d > best:
            best = d
            torch.save({"state_dict": model.state_dict(), "val_dice": d, "seed": seed}, path)
    (engine.RUNS_DIR / name).mkdir(parents=True, exist_ok=True)
    json.dump({"run": name, "best_val_dice": best}, open(engine.RUNS_DIR / name / "summary.json", "w"))
    return path


class Segmenter:
    def __init__(self, path=None):
        path = path or best_unet_path()
        self.dev = engine.device()
        self.model = UNetEffB0(pretrained=False)
        self.model.load_state_dict(torch.load(path, map_location="cpu", weights_only=False)["state_dict"])
        self.model.to(self.dev).eval()

    @torch.no_grad()
    def __call__(self, img: np.ndarray) -> np.ndarray:
        """Binary mask at the image's own resolution."""
        p = torch.sigmoid(self.model(_seg_input(img)[None].to(self.dev)))[0, 0].cpu().numpy()
        return cv2.resize((p > 0.5).astype(np.uint8), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)


def best_unet_path():
    best = max(config.SEEDS, key=lambda sd: torch.load(config.MODELS_DIR / f"unet_s{sd}.pth", map_location="cpu",
                                                       weights_only=False)["val_dice"])
    return config.MODELS_DIR / f"unet_s{best}.pth"


def otsu_mask(img):
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (11, 11), 0)
    _, m = cv2.threshold(g, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return m.astype(np.uint8)


def largest_component(mask):
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if n <= 1:
        return mask.astype(np.uint8)
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    return (lab == k).astype(np.uint8)


def crop_with_mask(img, mask, margin=config.CROP_MARGIN, min_area=config.CROP_MIN_AREA):
    """Square crop around the largest predicted region + margin; full image if region < 1%."""
    m = largest_component(mask)
    if m.mean() < min_area:
        return img, False
    ys, xs = np.nonzero(m)
    h, w = m.shape
    cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
    side = int(min(max(ys.max() - ys.min(), xs.max() - xs.min()) * (1 + 2 * margin), min(h, w)))
    side = max(side, 8)
    top = int(np.clip(cy - side / 2, 0, h - side)); left = int(np.clip(cx - side / 2, 0, w - side))
    return img[top:top + side, left:left + side], True


def segmentation_study():
    s = ham.splits()
    masks = find_masks()
    n_ham = sum(1 for k in masks if k in set(pd.concat(s.values()).image_id))
    print(f"HAM10000 masks found: {n_ham}")
    if n_ham < (50 if config.SMOKE else 5000):
        raise FileNotFoundError("HAM10000 lesion masks not found: add them under data/ham_masks or as a Kaggle input")
    for sd in config.SEEDS:
        train_unet(sd, s, masks)
    te = s["test"][s["test"].image_id.isin(masks)].reset_index(drop=True)
    g = te.lesion_id.to_numpy()
    per_seed, imgs, gts = [], [load_image_rgb(p) for p in te.path], []
    for p in te.image_id:
        gts.append(cv2.imread(str(masks[p]), cv2.IMREAD_GRAYSCALE) > 127)
    for sd in config.SEEDS:
        seg = Segmenter(config.MODELS_DIR / f"unet_s{sd}.pth")
        per_seed.append(np.array([_dice_iou(seg(i).astype(bool), gt) for i, gt in zip(imgs, gts)]))
    otsu = np.array([_dice_iou(otsu_mask(i).astype(bool), gt) for i, gt in zip(imgs, gts)])

    def seedmean_ci(col):
        return stats.bootstrap_ci(lambda ix: np.mean([ps[ix, col].mean() for ps in per_seed]), g)
    rows = [{"method": "U-Net (3 seeds)",
             "dice": f"{np.mean([ps[:, 0].mean() for ps in per_seed]):.3f} ± {np.std([ps[:, 0].mean() for ps in per_seed], ddof=1):.3f}",
             "dice_ci": seedmean_ci(0),
             "iou": f"{np.mean([ps[:, 1].mean() for ps in per_seed]):.3f} ± {np.std([ps[:, 1].mean() for ps in per_seed], ddof=1):.3f}",
             "iou_ci": seedmean_ci(1)},
            {"method": "Otsu baseline (no training)", "dice": f"{otsu[:, 0].mean():.3f}",
             "dice_ci": stats.bootstrap_ci(lambda ix: otsu[ix, 0].mean(), g),
             "iou": f"{otsu[:, 1].mean():.3f}", "iou_ci": stats.bootstrap_ci(lambda ix: otsu[ix, 1].mean(), g)}]
    pd.DataFrame(rows).to_csv(OUT / "table_6_12_segmentation.csv", index=False)
    print(pd.DataFrame(rows).to_string(index=False))

    seg = Segmenter()
    fig, axes = plt.subplots(3, 5, figsize=(13, 8))
    for i in range(min(5, len(te))):
        m = seg(imgs[i]); c, _ = crop_with_mask(imgs[i], m)
        ov = imgs[i].copy()
        cnts, _ = cv2.findContours(largest_component(m), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, cnts, -1, (0, 255, 0), 3)
        for row, (im, t) in enumerate(((imgs[i], "test image"), (ov, "predicted mask"), (c, "crop"))):
            axes[row, i].imshow(im); axes[row, i].set_title(t, fontsize=8); axes[row, i].axis("off")
    fig.tight_layout(); fig.savefig(OUT / "fig_4_4_masks_crops.png", dpi=150); plt.close(fig)


def make_crop_cache(df, src_col="path", cache_col="crop_cache", seg=None):
    seg = seg or Segmenter()
    used = 0
    for src, dst in zip(df[src_col], df[cache_col]):
        if Path(dst).exists():
            continue
        img = load_image_rgb(src)
        crop, ok = crop_with_mask(img, seg(img))
        used += ok
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        save_image_rgb(preprocess_image(crop), dst)
    return used


def crop_study():
    s = ham.splits()
    seg = Segmenter()
    for df in s.values():
        make_crop_cache(df, seg=seg)
    sel = selected_setup()
    for seed in config.SEEDS:
        engine.train_run(_run("crop", seed), ARCH, s["train"], s["val"], seed=seed, imbalance=sel,
                         cache_col="crop_cache")
    yt, gt = s["test"].label.to_numpy(), s["test"].lesion_id.to_numpy()
    base = [_preds(r, s["test"], "test") for r in baseline_runs()]
    crop = [_preds(_run("crop", sd), s["test"], "test", "crop_cache") for sd in config.SEEDS]
    _paired_table(yt, gt, base, crop, "Full image", "Crop", OUT / "table_6_13_crop")


# =========================================================================== robustness
def corrupt(img, kind, level):
    if kind == "brightness":
        return np.clip(img.astype(np.float32) * level, 0, 255).astype(np.uint8)
    if kind == "gaussian_blur":
        return cv2.GaussianBlur(img, (0, 0), level)
    if kind == "jpeg_quality":
        ok, buf = cv2.imencode(".jpg", cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(level)])
        return cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    raise ValueError(kind)


def _corrupt_one(item):
    src, dst, kind, level = item
    img = load_image_rgb(src)
    if kind != "clean":
        img = corrupt(img, kind, level)
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    save_image_rgb(preprocess_image(img), dst)


def robustness_study():
    from concurrent.futures import ProcessPoolExecutor
    s = ham.splits()
    te = s["test"].copy()
    yt, gt = te.label.to_numpy(), te.lesion_id.to_numpy()
    root = config.DATA_DIR / "corrupted"
    variants = [("clean", 0, None)] + [(k, i + 1, lv) for k, lvs in config.CORRUPTIONS.items() for i, lv in enumerate(lvs)]
    jobs = []
    for kind, sev, lv in variants:
        for src, iid in zip(te.path, te.image_id):
            dst = root / f"{kind}_{sev}" / f"{iid}.jpg"
            if not dst.exists():
                jobs.append((src, str(dst), kind, lv))
    if jobs:
        with ProcessPoolExecutor() as ex:
            list(ex.map(_corrupt_one, jobs, chunksize=32))
    wmap, _ = ham.ensemble_weights()
    models = {m: (config.MODELS_DIR / f"{m}.pth", True) for m in config.HAM_MODELS}
    models.update({f"study_s{sd}": (config.MODELS_DIR / f"{r}.pth", False) for sd, r in zip(config.SEEDS, baseline_runs())})
    rows = []
    for kind, sev, lv in variants:
        te["v"] = te.image_id.map(lambda i: str(root / f"{kind}_{sev}" / f"{i}.jpg"))
        P = {m: engine.cached_predictions(f"robust_{m}_{kind}_{sev}", path, te, "v", tta=tta)
             for m, (path, tta) in models.items()}
        P["ensemble"] = sum(w * P[m] for m, w in wmap.items())
        study = [P[f"study_s{sd}"] for sd in config.SEEDS]
        for m, p in list(P.items()) + [("study_setup", None)]:
            if m.startswith("study_s"):
                continue
            if m == "study_setup":
                ss = stats.seed_summary(yt, study, gt, ["macro_f1", "malignant_sensitivity", "accuracy"])
                r = {k: v["mean"] for k, v in ss.items()}
                ci = {k: v["ci"] for k, v in ss.items()}
            else:
                r = stats.metrics(yt, p)
                ci = {k: stats.bootstrap_ci(lambda ix, k=k: stats.metrics(yt[ix], p[ix])[k], gt, n=min(200, config.N_BOOTSTRAP))
                      for k in ("macro_f1", "malignant_sensitivity")}
            rows.append({"model": m, "corruption": kind, "severity": sev, "level": lv,
                         "macro_f1": r["macro_f1"], "malignant_sensitivity": r["malignant_sensitivity"],
                         "accuracy": r["accuracy"], "macro_f1_ci": ci["macro_f1"], "mal_sens_ci": ci["malignant_sensitivity"]})
    df = pd.DataFrame(rows)
    df.to_json(OUT / "robustness_all.json", orient="records", indent=1)
    clean = df[df.corruption == "clean"].set_index("model").macro_f1
    t = {"clean (severity 0)": clean}
    for kind in config.CORRUPTIONS:
        t[kind] = df[(df.corruption == kind) & (df.severity == 5)].set_index("model").macro_f1
    pd.DataFrame(t).T.to_csv(OUT / "table_6_14_robustness.csv")
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for j, kind in enumerate(config.CORRUPTIONS):
        for i, met in enumerate(("macro_f1", "malignant_sensitivity")):
            ax = axes[i, j]
            for m in ["ensemble", "study_setup"] + config.HAM_MODELS:
                d = pd.concat([df[(df.model == m) & (df.corruption == "clean")],
                               df[(df.model == m) & (df.corruption == kind)]]).sort_values("severity")
                ax.plot(d.severity, d[met], "o-", label=m, lw=2.5 if m in ("ensemble", "study_setup") else 1)
                if m in ("ensemble", "study_setup"):
                    cic = "macro_f1_ci" if met == "macro_f1" else "mal_sens_ci"
                    ax.fill_between(d.severity, [c[0] for c in d[cic]], [c[1] for c in d[cic]], alpha=.15)
            ax.set_title(f"{kind} - {met.replace('_', ' ')}"); ax.set_xlabel("severity"); ax.grid(alpha=.3)
    axes[0, 0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(OUT / "fig_6_8_robustness.png", dpi=150); plt.close(fig)
    print(pd.DataFrame(t).T.round(3).to_string())


# =========================================================================== explanations
@torch.no_grad()
def _curve(model, x, blurred, order, cls, steps, mode):
    n = order.numel()
    batch = []
    for k in range(steps + 1):
        idx = order[: int(round(n * k / steps))]
        m = torch.zeros(n, device=x.device); m[idx] = 1
        m = m.view(1, 1, *x.shape[-2:])
        batch.append(x * (1 - m) + blurred * m if mode == "deletion" else blurred * (1 - m) + x * m)
    p = torch.softmax(model(torch.cat(batch)), 1)[:, cls].float().cpu().numpy()
    return float(np.trapezoid(p, dx=1 / steps)) if hasattr(np, "trapezoid") else float(np.trapz(p, dx=1 / steps))


def explanation_study():
    from gradcam import GradCAM
    s = ham.splits()
    te = s["test"].sample(n=min(config.EXPLAIN_IMAGES, len(s["test"])), random_state=42).reset_index(drop=True)
    tf = engine.eval_transform()
    runs = {m: config.MODELS_DIR / f"{m}.pth" for m in
            ("efficientnet_b0", "resnet50", "densenet121", "convnext_tiny", "swin_t")}
    runs["study_setup_seed0"] = config.MODELS_DIR / f"{baseline_runs()[0]}.pth"
    rows = []
    rng = np.random.default_rng(0)
    for name, path in runs.items():
        model, ck = load_checkpoint(path)
        dev = next(model.parameters()).device
        gc = GradCAM(model, get_target_layer(model, ck.get("arch", name)))
        res = []
        for r in te.itertuples():
            img = load_image_rgb(r.cache)
            x = tf(img)[None].to(dev)
            blurred = tf(cv2.GaussianBlur(img, (0, 0), 10))[None].to(dev)
            cam, cls, _ = gc(x)
            order = torch.from_numpy(np.argsort(-cam.ravel())).to(dev)
            rnd = torch.from_numpy(rng.permutation(cam.size)).to(dev)
            res.append([_curve(model, x, blurred, order, cls, config.EXPLAIN_STEPS, "deletion"),
                        _curve(model, x, blurred, order, cls, config.EXPLAIN_STEPS, "insertion"),
                        _curve(model, x, blurred, rnd, cls, config.EXPLAIN_STEPS, "deletion"),
                        _curve(model, x, blurred, rnd, cls, config.EXPLAIN_STEPS, "insertion")])
        gc.remove()
        res = np.array(res)
        g = te.lesion_id.to_numpy()
        rows.append({"model": name,
                     "deletion": res[:, 0].mean(), "deletion_ci": stats.bootstrap_ci(lambda ix: res[ix, 0].mean(), g),
                     "insertion": res[:, 1].mean(), "insertion_ci": stats.bootstrap_ci(lambda ix: res[ix, 1].mean(), g),
                     "random_deletion": res[:, 2].mean(), "random_insertion": res[:, 3].mean()})
        print(rows[-1])
        del model
    pd.DataFrame(rows).to_csv(OUT / "table_6_15_explanations.csv", index=False)

In [ ]:
%%writefile src/dermacon.py
"""Indian-skin study on DermaCon-IN (report Sections 3.2, 4.5, 4.6, 6.2-6.4).

  audit()          Tables 3.2-3.6, Fig 3.3, patient-level split, configs/dermacon_mapping.csv
  zero_shot()      Tables 6.3-6.6, Fig 6.5   (HAM models + ensemble, no retraining)
  gate()           Tables 6.7-6.8, Fig 6.6   (Stage 1 EfficientNet-B0 on 8 main classes, 3 seeds)
  ddi()            Table 6.9                 (runs only if data/ddi/ddi_metadata.csv exists)

DermaCon-IN is research-use data obtained after approval: put your copy under data/dermacon/
(any folder layout: the metadata CSV/XLSX and the images are found automatically). Column names
are detected automatically and printed; if detection is wrong, write configs/dermacon_columns.json
e.g. {"image": "Image Name", "patient": "Patient ID", ...}.
"""
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.metrics import f1_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402
import engine  # noqa: E402
import ham  # noqa: E402
import stats  # noqa: E402

OUT = config.STUDY_DIR / "dermacon"
OUT.mkdir(parents=True, exist_ok=True)
SPLIT_FILES = {s: config.SPLITS_DIR / f"dermacon_{s}.csv" for s in ("train", "val", "test")}
COLS = ["image", "patient", "main_class", "sub_class", "diagnosis", "fst", "mst"]
KEYS = {
    "image": ["imagename", "imagefile", "filename", "imageid", "image", "img", "file"],
    "patient": ["patientid", "patient", "subjectid", "caseid", "pid"],
    "main_class": ["mainclass", "maincategory", "maindiagnosticclass", "main"],
    "sub_class": ["subclass", "subcategory", "etiology", "sub"],
    "diagnosis": ["diseaselabel", "disease", "diagnosis", "label"],
    "fst": ["fitzpatrick", "fst"],
    "mst": ["monk", "mst"],
}
MALIGNANT_RE = re.compile(r"melanoma|basal\s*cell|squamous\s*cell|bowen|keratoacanthoma", re.I)
MAPPING_RULES = [  # (regex on diagnosis, HAM10000 class); first match wins
    (r"(?<!non[- ])melanoma", "mel"), (r"basal\s*cell", "bcc"), (r"seborr?h?oeic\s*keratos|seborrheic\s*keratos", "bkl"),
    (r"nevocytic|melanocytic\s*nev|compound\s*nev|junctional\s*nev|intradermal\s*nev", "nv"),
    (r"dermatofibroma", "df"), (r"cherry\s*angioma|ha?emangioma|pyogenic\s*granuloma", "vasc"),
]


# --------------------------------------------------------------------------- loading
def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def find_metadata():
    roots = [config.DERMACON_DIR] + ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    best = None
    for r in roots:
        for f in list(r.rglob("*.csv")) + list(r.rglob("*.xlsx")):
            if "ham10000" in f.name.lower() or "isic" in f.name.lower():
                continue
            try:
                df = pd.read_csv(f, nrows=5) if f.suffix == ".csv" else pd.read_excel(f, nrows=5)
            except Exception:
                continue
            score = sum(any(k in _norm(c) for c in df.columns for k in KEYS[x][:2]) for x in COLS)
            if score < 3 or (best is not None and score < best[0]):
                continue
            # tie-break on row count: the full per-image metadata beats a subset (e.g. train/test split files)
            rows = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) if f.suffix == ".csv" else len(pd.read_excel(f))
            if best is None or (score, rows) > best[:2]:
                best = (score, rows, f)
    if best is None or best[0] < 3:
        raise FileNotFoundError("DermaCon-IN metadata not found. Put the dataset under data/dermacon/ "
                                "(or attach it as a Kaggle input).")
    return best[2]


def detect_columns(df) -> dict:
    override = config.CONFIGS_DIR / "dermacon_columns.json"
    if override.exists():
        return json.load(open(override))
    found, used = {}, set()
    for col in COLS:
        for key in KEYS[col]:
            hit = [c for c in df.columns if key in _norm(c) and c not in used]
            if hit:
                found[col] = hit[0]; used.add(hit[0]); break
    return found


def load() -> pd.DataFrame:
    f = find_metadata()
    raw = pd.read_csv(f) if f.suffix == ".csv" else pd.read_excel(f)
    cols = detect_columns(raw)
    print(f"DermaCon-IN metadata: {f}\ncolumns in file: {list(raw.columns)}\ndetected: {cols}")
    missing = [c for c in ("image", "patient", "main_class") if c not in cols]
    if missing:
        raise KeyError(f"could not detect columns {missing}; write configs/dermacon_columns.json")
    df = pd.DataFrame({c: raw[cols[c]] if c in cols else np.nan for c in COLS})
    root = f.parent
    for _ in range(3):        # images may sit next to the CSV or a few folders above it
        if any(root.rglob("*.jp*g")) or any(root.rglob("*.png")) or root.parent in (root, Path("/kaggle/input"), config.DATA_DIR):
            break
        root = root.parent
    files = {}
    for p in list(root.rglob("*.jpg")) + list(root.rglob("*.jpeg")) + list(root.rglob("*.png")) + list(root.rglob("*.JPG")):
        files.setdefault(p.name.lower(), p); files.setdefault(p.stem.lower(), p)
    df["path"] = df["image"].map(lambda v: files.get(str(v).lower(), files.get(Path(str(v)).stem.lower())))
    n_miss = df.path.isna().sum()
    if n_miss:
        print(f"warning: {n_miss} metadata rows without an image file - dropped")
    df = df.dropna(subset=["path"]).reset_index(drop=True)
    df["path"] = df.path.astype(str)
    df["image_id"] = df["path"].map(lambda p: "dc_" + re.sub(r"[^A-Za-z0-9_-]", "_", Path(p).stem))
    df["cache"] = df.image_id.map(lambda i: str(config.PROCESSED_DERMACON_DIR / f"{i}.jpg"))
    df["crop_cache"] = df.image_id.map(lambda i: str(config.PROCESSED_DERMACON_CROP_DIR / f"{i}.jpg"))
    df["main_class"] = df.main_class.astype(str).str.strip()
    df["neoplasm"] = df.main_class.str.contains("neoplas", case=False)
    df["malignant"] = df.diagnosis.astype(str).str.contains(MALIGNANT_RE)
    df["ham_class"] = df.diagnosis.astype(str).map(map_to_ham)
    df["group"] = [disease_group(m, s, d) for m, s, d in zip(df.main_class, df.sub_class, df.diagnosis)]
    df["fst_num"] = df.fst.map(parse_scale)
    df["mst_num"] = df.mst.map(parse_scale)
    return df


def map_to_ham(diag: str) -> str:
    for rx, c in MAPPING_RULES:
        if re.search(rx, str(diag), re.I):
            return c
    return "unmapped"


def disease_group(main, sub, diag) -> str:
    t = f"{sub} {diag}".lower()
    m = str(main).lower()
    if "infect" in m:
        for k, g in (("fung", "Fungal infection"), ("tinea", "Fungal infection"), ("bacter", "Bacterial infection"),
                     ("viral", "Viral infection"), ("wart", "Viral infection"), ("parasit", "Parasitic (scabies etc.)"),
                     ("scabies", "Parasitic (scabies etc.)")):
            if k in t:
                return g
    if "inflam" in m:
        for k, g in (("eczema", "Eczema / dermatitis"), ("dermatitis", "Eczema / dermatitis"),
                     ("psoria", "Psoriasis / lichenoid"), ("lichen", "Psoriasis / lichenoid"),
                     ("papulosquam", "Psoriasis / lichenoid")):
            if k in t:
                return g
    if "append" in m:
        for k, g in (("acne", "Acne / sebaceous"), ("sebac", "Acne / sebaceous"), ("hair", "Hair disorders"),
                     ("alopecia", "Hair disorders")):
            if k in t:
                return g
    return str(main)


def parse_scale(v):
    if pd.isna(v):
        return np.nan
    s = str(v).upper().strip()
    roman = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6}
    m = re.search(r"\b(VI|IV|V|III|II|I)\b", s)
    if m and not re.search(r"\d", s):
        return roman[m.group(1)]
    m = re.search(r"\d+", s)
    return float(m.group()) if m else np.nan


# --------------------------------------------------------------------------- audit + split
def make_split(df, force=False):
    if all(f.exists() for f in SPLIT_FILES.values()) and not force:
        parts = {s: pd.read_csv(f) for s, f in SPLIT_FILES.items()}
        ids = {s: set(p.image_id) for s, p in parts.items()}
        return {s: df[df.image_id.isin(ids[s])].reset_index(drop=True) for s in parts}
    counts = df.main_class.value_counts()
    pat = df.groupby("patient").agg(mal=("malignant", "any"),
                                    rare=("main_class", lambda x: min(x, key=lambda c: counts[c]))).reset_index()
    pat["stratum"] = np.where(pat.mal, "malignant", pat.rare)
    small = pat.stratum.value_counts()
    pat.loc[pat.stratum.map(small) < 6, "stratum"] = "rare"     # tiny strata merged so the split can stratify
    tr, rest = train_test_split(pat, test_size=0.30, stratify=pat.stratum, random_state=config.SEED)
    va, te = train_test_split(rest, test_size=0.50, stratify=rest.stratum, random_state=config.SEED)
    out = {}
    for s, p in (("train", tr), ("val", va), ("test", te)):
        out[s] = df[df.patient.isin(set(p.patient))].reset_index(drop=True)
        out[s][["image_id", "patient", "main_class"]].to_csv(SPLIT_FILES[s], index=False)
    assert not set(out["train"].patient) & set(out["test"].patient)
    assert not set(out["train"].patient) & set(out["val"].patient)
    assert not set(out["val"].patient) & set(out["test"].patient)
    assert out["test"].malignant.any(), "test split has no malignant image"
    return out


def splits():
    df = load()
    s = make_split(df)
    if config.SMOKE:
        s = {k: v.groupby("main_class").head(6 if k == "train" else 4).reset_index(drop=True) for k, v in s.items()}
        keep = set(s["test"].image_id) | set(df[df.malignant].image_id)
        s["test"] = df[df.image_id.isin(keep) & df.image_id.isin(pd.read_csv(SPLIT_FILES["test"]).image_id)].reset_index(drop=True)
    main = sorted(df.main_class.unique())
    for v in s.values():
        v["main_label"] = v.main_class.map({c: i for i, c in enumerate(main)})
    return s, main


def audit():
    df = load()
    s = make_split(df, force=not all(f.exists() for f in SPLIT_FILES.values()))
    t32 = df.groupby("main_class").agg(images=("image_id", "size"), patients=("patient", "nunique"))
    t32.sort_values("images", ascending=False).to_csv(OUT / "table_3_2_main_classes.csv")
    t33 = pd.DataFrame({k: v.main_class.value_counts() for k, v in s.items()}).fillna(0).astype(int)
    t33.loc["TOTAL images"] = [len(s[k]) for k in t33.columns]
    t33.loc["TOTAL patients"] = [s[k].patient.nunique() for k in t33.columns]
    t33.to_csv(OUT / "table_3_3_split.csv")
    mal = df[df.malignant].copy()
    mal["split"] = mal.image_id.map({i: k for k, v in s.items() for i in v.image_id})
    t34 = mal.pivot_table(index="diagnosis", columns="split", values="image_id", aggfunc="count", fill_value=0)
    t34["patients"] = mal.groupby("diagnosis").patient.nunique()
    t34.to_csv(OUT / "table_3_4_malignant.csv")
    df.groupby("ham_class").agg(images=("image_id", "size"), patients=("patient", "nunique")).to_csv(
        OUT / "table_3_5_mapping.csv")
    (df[["diagnosis", "ham_class"]].drop_duplicates().sort_values(["ham_class", "diagnosis"])
     .to_csv(config.CONFIGS_DIR / "dermacon_mapping.csv", index=False))
    t36 = pd.concat([df.fst_num.value_counts().sort_index().rename(lambda x: f"Fitzpatrick type {int(x)}"),
                     df.mst_num.value_counts().sort_index().rename(lambda x: f"Monk tone {int(x)}")])
    t36.to_csv(OUT / "table_3_6_skin_tone.csv", header=["images"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    df.fst_num.value_counts().sort_index().plot.bar(ax=axes[0], color="sienna"); axes[0].set_title("Fitzpatrick type")
    df.mst_num.value_counts().sort_index().plot.bar(ax=axes[1], color="peru"); axes[1].set_title("Monk skin tone")
    fig.tight_layout(); fig.savefig(OUT / "fig_3_3_skin_tone.png", dpi=150); plt.close(fig)
    df.groupby("group").size().to_csv(OUT / "disease_groups.csv", header=["images"])
    print(t32, t33, t34, sep="\n\n")
    engine.ensure_cached(pd.concat(splits()[0].values()) if config.SMOKE else df, "path", "cache")
    return df


# --------------------------------------------------------------------------- zero-shot
def _fcr_row(name, called, patients, n_boot_seed=0):
    k, n = int(called.sum()), len(called)
    lo, hi = stats.wilson(k, n)
    plo, phi = stats.bootstrap_ci(lambda ix: called[ix].mean(), patients, seed=n_boot_seed)
    return {"model": name, "called_cancer": k, "n": n, "fcr": k / max(n, 1), "wilson": [lo, hi], "patient_ci": [plo, phi]}


def ham_probs(df, key):
    """HAM model + ensemble probabilities (3-view TTA) for DermaCon-IN images."""
    P = {m: engine.cached_predictions(f"dc_{key}_{m}", config.MODELS_DIR / f"{m}.pth", df) for m in config.HAM_MODELS}
    wmap, _ = ham.ensemble_weights()
    P["ensemble"] = sum(w * P[m] for m, w in wmap.items())
    return P


def zero_shot():
    import studies
    s, _ = splits()
    te = s["test"]; allx = pd.concat(s.values()).reset_index(drop=True)
    nn_ = te[~te.neoplasm].reset_index(drop=True)
    P = ham_probs(te, "test")
    called = {m: np.isin(p.argmax(1), stats.MAL)[~te.neoplasm.to_numpy()] for m, p in P.items()}
    rows = [_fcr_row(m, c, nn_.patient.to_numpy()) for m, c in called.items()]

    def study_row(label, runs, cache_col="cache"):
        cs = [np.isin(engine.cached_predictions(f"dc_test_{r}", config.MODELS_DIR / f"{r}.pth", nn_, cache_col,
                                                tta=False).argmax(1), stats.MAL) for r in runs]
        rates = [c.mean() for c in cs]
        ci = stats.bootstrap_ci(lambda ix: np.mean([c[ix].mean() for c in cs]), nn_.patient.to_numpy())
        return {"model": label, "fcr": float(np.mean(rates)), "fcr_std": float(np.std(rates, ddof=1)),
                "per_seed": rates, "patient_ci": ci}
    base_runs = studies.baseline_runs()
    rows.append(study_row("Selected study setup (3 seeds)", base_runs))
    rows.append(study_row("Skin-tone augmented (3 seeds)", [f"study_skin_s{sd}" for sd in config.SEEDS]))
    if (config.MODELS_DIR / "study_crop_s0.pth").exists():
        studies.make_crop_cache(nn_, seg=studies.Segmenter())
        rows.append(study_row("Crop model (3 seeds)", [f"study_crop_s{sd}" for sd in config.SEEDS], "crop_cache"))
    # BCCs called cancer by the study models (text of Section 6.6)
    mal_te = te[te.malignant].reset_index(drop=True)
    extra = {}
    for label, runs in (("study", base_runs), ("skin", [f"study_skin_s{sd}" for sd in config.SEEDS])):
        extra[label] = [int(np.isin(engine.cached_predictions(f"dc_testmal_{r}", config.MODELS_DIR / f"{r}.pth", mal_te,
                                                                tta=False).argmax(1), stats.MAL).sum()) for r in runs]
    json.dump({"rows": rows, "test_malignant_called_by_study_models": extra, "n_test_malignant": len(mal_te)},
              open(OUT / "table_6_3_fcr.json", "w"), indent=2, default=float)
    pd.DataFrame(rows).to_csv(OUT / "table_6_3_fcr.csv", index=False)

    pe = P["ensemble"][~te.neoplasm.to_numpy()]
    top = pd.Series([config.CLASS_NAMES[i] for i in pe.argmax(1)]).value_counts()
    top.to_csv(OUT / "ensemble_top_class_non_neoplastic.csv", header=["images"])
    ce = called["ensemble"]
    grp = []
    for g, ix in nn_.groupby("group").groups.items():
        ix = np.asarray(list(ix)); k, n = int(ce[ix].sum()), len(ix); lo, hi = stats.wilson(k, n)
        grp.append({"group": g, "images": n, "called_cancer": k, "fcr": k / n, "wilson_lo": lo, "wilson_hi": hi})
    grp = pd.DataFrame(grp).sort_values("fcr", ascending=False)
    grp.to_csv(OUT / "table_6_4_groups.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(grp) + 1))
    ax.errorbar(grp.fcr, range(len(grp)), xerr=[grp.fcr - grp.wilson_lo, grp.wilson_hi - grp.fcr], fmt="o", capsize=3)
    ax.set_yticks(range(len(grp))); ax.set_yticklabels(grp.group); ax.invert_yaxis()
    ax.set_xlabel("false-cancer rate (95% Wilson CI)"); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUT / "fig_6_5_fcr_groups.png", dpi=150); plt.close(fig)

    tone = []
    for scale, col, bins in (("Fitzpatrick", "fst_num", [("FST III-IV", 3, 4), ("FST V-VI", 5, 6)]),
                             ("Monk", "mst_num", [("MST 1-5", 1, 5), ("MST 6", 6, 6), ("MST 7", 7, 7), ("MST 8-10", 8, 10)])):
        v = nn_[col].to_numpy()
        for name, lo_, hi_ in bins:
            sel = (v >= lo_) & (v <= hi_)
            k, n = int(ce[sel].sum()), int(sel.sum()); lo, hi = stats.wilson(k, n)
            tone.append({"scale": scale, "group": name, "images": n, "fcr": k / max(n, 1), "wilson_lo": lo, "wilson_hi": hi})
    pd.DataFrame(tone).to_csv(OUT / "table_6_5_skin_tone.csv", index=False)

    Pall = ham_probs(allx, "all")
    mapped = te[te.ham_class != "unmapped"]
    Pm = ham_probs(mapped.reset_index(drop=True), "mapped") if len(mapped) else None
    sec = []
    for m in P:
        acc = float((Pm[m].argmax(1) == mapped.ham_class.map(config.CLASS_NAMES.index).to_numpy()).mean()) if Pm else float("nan")
        mal = allx.malignant.to_numpy()
        det = np.isin(Pall[m].argmax(1), stats.MAL)
        k, n = int(det[mal].sum()), int(mal.sum()); lo, hi = stats.wilson(k, n)
        sec.append({"model": m, "mapped_acc_test": acc, "n_mapped_test": len(mapped), "malignant_detected": k / max(n, 1),
                    "malignant_k": k, "malignant_n": n, "wilson_lo": lo, "wilson_hi": hi,
                    "fcr_all_images": float(det[~allx.neoplasm.to_numpy()].mean())})
    pd.DataFrame(sec).to_csv(OUT / "table_6_6_secondary.csv", index=False)
    print(pd.DataFrame(rows)[["model", "fcr"]].to_string(index=False))


# --------------------------------------------------------------------------- gate / two-stage
def gate():
    s, main = splits()
    neo = [i for i, c in enumerate(main) if "neoplas" in c.lower()][0]
    for sd in config.SEEDS:
        engine.train_run(f"gate_s{sd}", "efficientnet_b0", s["train"], s["val"], n_classes=len(main),
                         class_names=main, label_col="main_label", seed=sd, imbalance="sqrt",
                         settings=config.TRAIN_SETTINGS["efficientnet_b0"])
    te, va = s["test"], s["val"]
    nonneo = ~te.neoplasm.to_numpy()
    pe = ham_probs(te, "test")["ensemble"]
    ens_called = np.isin(pe.argmax(1), stats.MAL)
    rows, two_called, thresholds = [], [], {}
    for sd in config.SEEDS:
        ck = config.MODELS_DIR / f"gate_s{sd}.pth"
        pv = engine.cached_predictions(f"gate_s{sd}_val", ck, va)
        pt = engine.cached_predictions(f"gate_s{sd}_test", ck, te)
        vneo = np.sort(pv[va.neoplasm.to_numpy(), neo])[::-1]
        k = int(np.ceil(config.GATE_NEOPLASM_RECALL * len(vneo)))
        t = float(vneo[k - 1]) if len(vneo) else 0.5           # highest t keeping >= 90% of val neoplasms
        thresholds[sd] = t
        passed = pt[:, neo] >= t
        yt = te.main_label.to_numpy()
        tn = te.neoplasm.to_numpy()
        rows.append({"seed": sd, "accuracy": float((pt.argmax(1) == yt).mean()),
                     "macro_f1": float(f1_score(yt, pt.argmax(1), average="macro", labels=list(range(len(main))), zero_division=0)),
                     "val_macro_f1": float(f1_score(va.main_label, pv.argmax(1), average="macro", labels=list(range(len(main))), zero_division=0)),
                     "neoplasm_recall": float((pt.argmax(1)[tn] == neo).mean()) if tn.any() else float("nan"),
                     "threshold": t, "test_neoplasms_passed": f"{int(passed[tn].sum())}/{int(tn.sum())}",
                     "non_neoplastic_passed": float(passed[nonneo].mean())})
        two_called.append((passed & ens_called)[nonneo])
    pd.DataFrame(rows).to_csv(OUT / "table_6_7_gate.csv", index=False)

    pats = te.patient.to_numpy()[nonneo]
    alone = ens_called[nonneo]
    rates = [c.mean() for c in two_called]
    ci_two = stats.bootstrap_ci(lambda ix: np.mean([c[ix].mean() for c in two_called]), pats)
    ci_alone = stats.bootstrap_ci(lambda ix: alone[ix].mean(), pats)
    ci_diff = stats.bootstrap_ci(lambda ix: np.mean([c[ix].mean() for c in two_called]) - alone[ix].mean(), pats)
    mal = te.malignant.to_numpy()
    bcc_alone = int(ens_called[mal].sum())
    bcc_two = []
    for sd in config.SEEDS:
        pt = engine.cached_predictions(f"gate_s{sd}_test", config.MODELS_DIR / f"gate_s{sd}.pth", te)
        bcc_two.append(int(((pt[:, neo] >= thresholds[sd]) & ens_called)[mal].sum()))
    t68 = {"ensemble_alone": {"fcr": float(alone.mean()), "patient_ci": ci_alone, "malignant_called": bcc_alone},
           "two_stage": {"fcr_mean": float(np.mean(rates)), "fcr_std": float(np.std(rates, ddof=1)), "per_seed": rates,
                         "patient_ci": ci_two, "malignant_called_per_seed": bcc_two},
           "change": {"points": float(np.mean(rates) - alone.mean()), "patient_ci": ci_diff},
           "n_test_malignant": int(mal.sum()), "thresholds": thresholds}

    # gate on HAM10000 dermoscopic validation images (best seed by DermaCon val macro-F1)
    best = max(rows, key=lambda r: r["val_macro_f1"])["seed"]
    hv = ham.splits()["val"]
    ph = engine.cached_predictions(f"gate_s{best}_hamval", config.MODELS_DIR / f"gate_s{best}.pth", hv)
    hp = ph[:, neo] >= thresholds[best]
    t68["gate_on_ham_val"] = {"seed": best, "passed_all": float(hp.mean()),
                              "passed_malignant": float(hp[hv.dx.isin(config.MALIGNANT_CLASSES).to_numpy()].mean())}
    json.dump(t68, open(OUT / "table_6_8_two_stage.json", "w"), indent=2, default=float)

    per = []
    for c in sorted(te.main_class.unique()):
        sel = (te.main_class == c).to_numpy()[nonneo]
        if sel.any():
            per.append({"main_class": c, "alone": alone[sel].mean(), "two_stage": np.mean([tc[sel].mean() for tc in two_called])})
    per = pd.DataFrame(per)
    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(per))
    ax.bar(x - .2, per.alone, .4, label="ensemble alone"); ax.bar(x + .2, per.two_stage, .4, label="two-stage (mean of 3 seeds)")
    ax.set_xticks(x); ax.set_xticklabels(per.main_class, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("false-cancer rate"); ax.legend(); ax.grid(axis="y", alpha=.3)
    fig.tight_layout(); fig.savefig(OUT / "fig_6_6_two_stage.png", dpi=150); plt.close(fig)
    print(pd.DataFrame(rows).to_string(index=False)); print(json.dumps(t68, indent=1, default=float))


# --------------------------------------------------------------------------- DDI
def ddi():
    meta = config.DDI_DIR / "ddi_metadata.csv"
    out = OUT / "table_6_9_ddi.json"
    if not meta.exists():
        json.dump({"status": "PENDING: DDI", "reason": "data/ddi/ddi_metadata.csv not found"}, open(out, "w"), indent=2)
        print("DDI not available - Table 6.9 stays [PENDING: DDI]")
        return
    df = pd.read_csv(meta)
    files = {p.name: p for p in config.DDI_DIR.rglob("*.png")} | {p.name: p for p in config.DDI_DIR.rglob("*.jpg")}
    df["path"] = df.DDI_file.map(lambda f: str(files.get(f, "")))
    df = df[df.path != ""].reset_index(drop=True)
    df["image_id"] = "ddi_" + df.DDI_file.str.replace(r"\W", "_", regex=True)
    df["cache"] = df.image_id.map(lambda i: str(config.DATA_DIR / "processed_ddi" / f"{i}.jpg"))
    engine.ensure_cached(df, "path", "cache")
    pe = ham_probs(df, "ddi")["ensemble"]
    y = df.malignant.astype(str).str.lower().isin(["true", "1"]).to_numpy()
    called = np.isin(pe.argmax(1), stats.MAL)
    rows = []
    for g, sel in [("FST I-II", df.skin_tone == 12), ("FST III-IV", df.skin_tone == 34), ("FST V-VI", df.skin_tone == 56),
                   ("All", np.ones(len(df), bool))]:
        sel = np.asarray(sel)
        ys, cs = y[sel], called[sel]
        se = stats.wilson(int(cs[ys].sum()), int(ys.sum())); sp = stats.wilson(int((~cs[~ys]).sum()), int((~ys).sum()))
        rows.append({"group": g, "images": int(sel.sum()), "malignant": int(ys.sum()),
                     "sensitivity": cs[ys].mean() if ys.any() else np.nan, "sens_ci": se,
                     "specificity": (~cs[~ys]).mean() if (~ys).any() else np.nan, "spec_ci": sp,
                     "auc": stats._safe_auc(ys, pe[sel][:, stats.MAL].sum(1))})
    json.dump({"status": "done", "rows": rows}, open(out, "w"), indent=2, default=float)
    print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
%%writefile src/sanity.py
"""Automatic sanity checks (report Section 5.6): notebook 09 (HAM10000) and notebook 19 (Indian study).
Each check returns (name, passed, detail); results go to results/study/sanity_*.json.
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))
import config  # noqa: E402

S = config.STUDY_DIR


def _save(name, checks):
    out = [{"check": c, "passed": bool(p), "detail": str(d)} for c, p, d in checks]
    json.dump(out, open(S / f"sanity_{name}.json", "w"), indent=2)
    n_ok = sum(c["passed"] for c in out)
    for c in out:
        print(("PASS " if c["passed"] else "FAIL ") + c["check"] + "  " + c["detail"])
    print(f"{n_ok}/{len(out)} checks passed")
    return out


def ham_checks():
    import ham
    s = ham.splits()
    L = {k: set(v.lesion_id) for k, v in s.items()}
    checks = [("no lesion in train and val", not L["train"] & L["val"], len(L["train"] & L["val"])),
              ("no lesion in train and test", not L["train"] & L["test"], len(L["train"] & L["test"])),
              ("no lesion in val and test", not L["val"] & L["test"], len(L["val"] & L["test"]))]
    n = sum(len(pd.read_csv(config.SPLITS_DIR / f"{k}.csv")) for k in ("train", "val", "test"))
    checks.append(("split covers all 10,015 images", n == 10015, n))
    c = json.load(open(S / "ham" / "ensemble_choice.json"))
    P = ham.load_preds()
    yv = s["val"].label.to_numpy()
    V = np.stack([P[m]["val"] for m in c["models"]])
    f = {k: f1_score(yv, np.tensordot(np.array(w), V, 1).argmax(1), average="macro", labels=list(range(7)), zero_division=0)
         for k, w in c["candidates"].items()}
    checks.append(("ensemble choice reproduced from validation", max(f, key=f.get) == c["chosen"], c["chosen"]))
    ok = all((config.MODELS_DIR / f"{m}.pth").exists() and len(P[m]["test"]) == len(s["test"]) for m in config.HAM_MODELS)
    checks.append(("six HAM10000 models trained and evaluated on the full test split", ok, len(s["test"])))
    return _save("ham", checks)


def indian_checks():
    import dermacon
    import studies
    checks = []
    parts = {k: pd.read_csv(f) for k, f in dermacon.SPLIT_FILES.items()}
    P = {k: set(v.patient) for k, v in parts.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        checks.append((f"no patient in DermaCon-IN {a} and {b}", not P[a] & P[b], len(P[a] & P[b])))
    df = dermacon.load()
    mal_test = df[df.image_id.isin(parts["test"].image_id) & df.malignant]
    checks.append(("DermaCon-IN test split contains malignant images", len(mal_test) > 0, len(mal_test)))
    for prefix, n in (("study_imb_", 15), ("study_skin_", 3), ("study_crop_", 3), ("unet_", 3), ("gate_", 3)):
        k = len(list(config.MODELS_DIR.glob(f"{prefix}*.pth")))
        checks.append((f"{prefix}* runs complete ({n} expected)", k >= n, k))
    sel = json.load(open(S / "studies" / "imbalance_selected.json"))
    val = {k: tuple(v) for k, v in sel["val"].items()}
    best = max(v[0] for v in val.values())
    re_sel = max([k for k, v in val.items() if best - v[0] <= 0.005], key=lambda k: (val[k][1], val[k][0]))
    checks.append(("imbalance setup reproduced from validation", re_sel == sel["selected"], sel["selected"]))
    t68 = json.load(open(S / "dermacon" / "table_6_8_two_stage.json"))
    checks.append(("gate thresholds stored for 3 seeds", len(t68["thresholds"]) == 3, t68["thresholds"]))
    fcr = json.load(open(S / "dermacon" / "table_6_3_fcr.json"))["rows"]
    n_nonneo = int((~df[df.image_id.isin(parts["test"].image_id)].neoplasm).sum())
    checks.append(("FCR denominator = non-neoplastic test images", all(r.get("n", n_nonneo) == n_nonneo for r in fcr), n_nonneo))
    checks.append(("label mapping file written", (config.CONFIGS_DIR / "dermacon_mapping.csv").exists(), "configs/dermacon_mapping.csv"))
    import torch
    seeds_ok = all(torch.load(f, map_location="cpu", weights_only=False).get("seed") in config.SEEDS
                   for f in config.MODELS_DIR.glob("study_*.pth"))
    checks.append(("every study checkpoint was trained with seed 0, 1 or 2", seeds_ok, "study_*.pth"))
    nums = S / "report_numbers.json"
    checks.append(("report numbers file written", nums.exists(), nums.name))
    checks.append(("not a smoke run", not config.SMOKE, f"SMOKE={config.SMOKE}"))
    return _save("indian", checks)

In [ ]:
%%writefile src/report_numbers.py
"""Collect every table produced by the notebooks into results/study/report_numbers.json,
so each number in the report can be traced to one file (report Section 1.5)."""
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
import config  # noqa: E402


def collect():
    out = {}
    for f in sorted(config.STUDY_DIR.rglob("*")):
        if f.name == "report_numbers.json" or f.suffix not in (".csv", ".json"):
            continue
        key = f.relative_to(config.STUDY_DIR).as_posix()
        out[key] = json.load(open(f)) if f.suffix == ".json" else pd.read_csv(f).to_dict(orient="records")
    runs = {}
    for f in sorted((config.RESULTS_DIR / "runs").glob("*/summary.json")):
        runs[f.parent.name] = json.load(open(f))
    out["training_runs"] = runs
    t = config.STUDY_DIR / "timings.json"
    out["notebook_minutes"] = json.load(open(t)) if t.exists() else {}
    json.dump(out, open(config.STUDY_DIR / "report_numbers.json", "w"), indent=1, default=str)
    print(f"{len(out)} entries -> {config.STUDY_DIR / 'report_numbers.json'}")
    return out


class timed:
    """with timed("14_imbalance_study"): ...  -> minutes stored in results/study/timings.json"""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        import time
        self.t0 = time.time()

    def __exit__(self, *a):
        import time
        f = config.STUDY_DIR / "timings.json"
        d = json.load(open(f)) if f.exists() else {}
        d[self.name] = round((time.time() - self.t0) / 60, 1)
        json.dump(d, open(f, "w"), indent=2)

In [ ]:
# HAM10000 input + the exact lesion-level split used in the report (Table 3.1)
import base64, gzip, pandas as pd
meta = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)
assert meta, "Add the 'skin-cancer-mnist-ham10000' dataset as an input"
raw = os.path.dirname(meta[0])
if not os.path.islink("data/raw"):
    shutil.rmtree("data/raw", ignore_errors=True); os.symlink(raw, "data/raw")
paths = {}
for p in sorted(glob.glob(f"{raw}/**/ISIC_*.jpg", recursive=True)):
    paths.setdefault(os.path.basename(p)[:-4], os.path.relpath(p, raw))
assign = dict(l.split(",") for l in gzip.decompress(base64.b64decode("...")).decode().splitlines())
md = pd.read_csv(meta[0])
classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
md["path"] = md.image_id.map(paths); md["label"] = md.dx.map({c: i for i, c in enumerate(classes)})
md["binary"] = md.dx.map(lambda d: "Malignant" if d in ("mel", "bcc", "akiec") else "Benign")
md["split"] = md.image_id.map(assign)
cols = ["lesion_id", "image_id", "dx", "dx_type", "age", "sex", "localization", "path", "label", "binary"]
for s, c in (("train", "r"), ("val", "v"), ("test", "e")):
    md[md.split == c][cols].to_csv(f"data/splits/{s}.csv", index=False)
print({s: len(pd.read_csv(f"data/splits/{s}.csv")) for s in ("train", "val", "test")})
sys.path[:0] = [WORK, f"{WORK}/src"]
from report_numbers import timed
FAILED = []
def step(name, fn):
    print(f"\n========== {name} ==========", flush=True)
    t0 = time.time()
    try:
        with timed(name):
            fn()
        print(f"[ok] {name} ({(time.time() - t0) / 60:.1f} min)")
    except Exception:
        traceback.print_exc(); FAILED.append(name)
        print(f"[FAILED] {name} - continuing with the next step")

In [ ]:
# bring in the outputs of the earlier parts (added as inputs: Add Input -> Your Work -> Notebooks)
found = [d for d in glob.glob("/kaggle/input/**/skin-cancer-detection", recursive=True) if os.path.isdir(d + "/models")]
print("restoring from:", found)
for d in found:
    for sub in ("models", "results", "configs"):
        if os.path.isdir(f"{d}/{sub}"):
            shutil.copytree(f"{d}/{sub}", sub, dirs_exist_ok=True)
timings = {}
for d in found:
    f = f"{d}/results/study/timings.json"
    if os.path.exists(f):
        timings.update(json.load(open(f)))
os.makedirs("results/study", exist_ok=True)
json.dump(timings, open("results/study/timings.json", "w"), indent=2)
print(len(glob.glob("models/*.pth")), "checkpoints available; timings so far:", timings)

In [ ]:
step("01_data", lambda: __import__('ham').prepare())

In [ ]:
step("17_robustness", lambda: __import__('studies').robustness_study())

In [ ]:
step("18_explainability", lambda: __import__('studies').explanation_study())

In [ ]:
step("10_indian_data_audit", lambda: __import__('dermacon').audit())

In [ ]:
step("11_indian_zero_shot", lambda: __import__('dermacon').zero_shot())

In [ ]:
step("12_stage1_gate", lambda: __import__('dermacon').gate())

In [ ]:
step("13_ddi_evaluation", lambda: __import__('dermacon').ddi())

In [ ]:
step("19_sanity_checks", lambda: (__import__('report_numbers').collect(), __import__('sanity').indian_checks()))

In [ ]:
import report_numbers
report_numbers.collect()
os.chdir("/kaggle/working")
subprocess.run("cd skin-cancer-detection && zip -qr /kaggle/working/part_C_outputs.zip results configs data/splits models", shell=True)
print("FAILED STEPS:", FAILED or "none")
if FAILED:
    raise RuntimeError(f"steps failed: {FAILED} - send the log")

In [ ]:
# ---- save the result to Google Drive (and offer a direct download) ----
os.makedirs("/content/drive/MyDrive/skin-cancer-part-C", exist_ok=True)
shutil.copy("/kaggle/working/part_C_outputs.zip", "/content/drive/MyDrive/skin-cancer-part-C/part_C_outputs.zip")
print("saved to MyDrive/skin-cancer-part-C/part_C_outputs.zip")
